# Pré-processamento

**Entrega: 07/11/2026 | Valor: 4,00 pontos.**

**Equipe:** Grupo 10

**Integrantes:**
- Samuel Lucas Rodrigues Vieira
- Matheus de Almeida Moreira
- Carlos Eduardo de Melo Sabino
- Arthur Soares Rolla
- Paulo Gabriel de Oliveira Leite

Este notebook apresenta os experimentos de pré-processamento da etapa 2. As quatro primeiras partes investigam limpeza, balanceamento, codificação, transformação numérica e redução de dimensionalidade. A quinta parte combina essas propostas, compara os resultados e registra as configurações encaminhadas à etapa 3.

As seções seguem a distribuição informada pela equipe: Samuel, Matheus, Carlos Eduardo, Arthur e Paulo. Os resultados apresentados são de validação interna no treino rotulado. O teste oficial não possui rótulos disponíveis para a equipe.


## Protocolo comum aos experimentos

- Usar os CSVs oficiais em `data/`: `treino.csv`, `teste_sem_rotulo.csv` e `modelo_de_submissao.csv`.
- O alvo é `class`, com valores `normal` e `anomaly`. Não remapear para cinco famílias de ataques.
- Não substituir, reembaralhar ou recriar a separação oficial de treino/teste. Validação interna é permitida **somente dentro do treino**.
- Ajustar imputação, codificação, normalização, balanceamento e redução apenas no treino de cada dobra.
- `id` identifica o teste e não é atributo preditivo. O teste não tem gabarito disponível à equipe.
- Investigar a associação de **cada atributo** com a classe antes de treinar modelos. Correlação alta não prova vazamento sozinha.
- Examinar também as colunas adicionadas, inclusive `score_ids_legado`, `tipo_alerta_soc` e `tempo_resposta_analista_min`. Justificar decisões pela origem e disponibilidade da informação.
- Não reutilizar conclusões ou números da base antiga. Não procurar/reconstruir rótulos do teste público.
- Os módulos em `src/` são a estrutura legada e precisam ser adaptados pelos responsáveis. Este notebook não utiliza esses módulos.
- Cada membro deve produzir código comentado, resultados e uma conclusão autoral em sua seção, além de revisar a integração.


## Samuel Lucas Rodrigues Vieira — Ausentes, redundâncias e inconsistências

**Objetivo da seção:** Implementar a carga comum e as estratégias indicadas pela etapa 1. Comparar tratamento de ausentes, normalização de categorias e manutenção/remoção de redundâncias com uma referência. Definir tratamento de duplicatas/conflitos sem contaminar validação e sem alterar o teste oficial.

**Conteúdo apresentado:** Transformações reutilizáveis e tabela de estratégias, evidências e justificativas. Nenhuma mediana ou decisão estatística aprendida fora das dobras.


### 1.1 Carga comum

Funções `carregar_treino()` e `carregar_teste()` para todos os membros. Os CSVs oficiais não são alterados, o alvo vira `y` (1 = `anomaly`), e `id` nunca entra como atributo.
Os sinais operacionais (`score_ids_legado`, `tipo_alerta_soc`, `tempo_resposta_analista_min`) ficam **fora** dos preditores por padrão, porque a etapa 1 levantou dúvidas sobre sua disponibilidade no momento da previsão. A integração da parte 5 mantém a exclusão preventiva desses sinais. Sua origem e disponibilidade temporal ainda precisam ser confirmadas com a professora. O argumento `incluir_operacionais=True` permite carregá-los para uma investigação específica, mas não faz parte da configuração recomendada.

In [1]:
# Samuel Lucas Rodrigues Vieira — 1.1 Carga comum (reutilizável pelos demais membros)
from pathlib import Path
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

SEMENTE = 42
N_DOBRAS = 5

project_root = Path.cwd().resolve()
if not (project_root / "data" / "treino.csv").exists():
    project_root = project_root.parent
if not (project_root / "data" / "treino.csv").exists():
    raise FileNotFoundError("Execute na raiz do projeto ou em notebooks/.")
DIR_DADOS = project_root / "data"

COL_ROTULO = "class"
COL_ID = "id"
CLASSES = ["normal", "anomaly"]          # anomaly = classe positiva (1)
COLS_CATEGORICAS = ["protocol_type", "service", "flag"]
# Sinais operacionais: ficam fora dos preditores até a conclusão do Paulo Gabriel de Oliveira Leite (etapa 1, seção 1.7).
SINAIS_OPERACIONAIS = ["score_ids_legado", "tipo_alerta_soc", "tempo_resposta_analista_min"]
# Colunas derivadas -> coluna de origem e relação esperada (verificadas na seção 1.2).
COLS_DERIVADAS = {
    "src_bytes_kb": ("src_bytes", "src_bytes / 1024"),
    "dst_bytes_kb": ("dst_bytes", "dst_bytes / 1024"),
    "total_bytes": ("src_bytes + dst_bytes", "src_bytes + dst_bytes"),
    "same_srv_pct": ("same_srv_rate", "same_srv_rate * 100"),
    "protocolo_cod": ("protocol_type", "código 1:1 de protocol_type"),
}
# Limites de domínio da documentação do KDD (conhecimento prévio, não aprendidos dos dados).
LIMITES_DOMINIO = {
    **{c: (0, 1) for c in ["serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
                           "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
                           "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
                           "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
                           "dst_host_serror_rate", "dst_host_srv_serror_rate",
                           "dst_host_rerror_rate", "dst_host_srv_rerror_rate"]},
    "same_srv_pct": (0, 100),
    "count": (0, 511), "srv_count": (0, 511),
    "dst_host_count": (0, 255), "dst_host_srv_count": (0, 255),
    "duration": (0, np.inf), "src_bytes": (0, np.inf), "dst_bytes": (0, np.inf),
    "total_bytes": (0, np.inf), "src_bytes_kb": (0, np.inf), "dst_bytes_kb": (0, np.inf),
}


def carregar_treino(incluir_operacionais: bool = False):
    """Lê treino.csv sem alterá-lo. Retorna (X, y) com y = 1 para anomaly."""
    df = pd.read_csv(DIR_DADOS / "treino.csv")
    assert set(df[COL_ROTULO]) == set(CLASSES) and df[COL_ROTULO].notna().all()
    assert COL_ID not in df.columns, "id não é atributo preditivo."
    descartar = [COL_ROTULO] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    X = df.drop(columns=descartar)
    y = (df[COL_ROTULO] == "anomaly").astype(int).rename(COL_ROTULO)
    return X, y


def carregar_teste(incluir_operacionais: bool = False):
    """Lê teste_sem_rotulo.csv somente para checar esquema/aplicar o pipeline já ajustado.

    Nenhuma estatística é calculada com o teste; ordem e IDs são preservados.
    """
    df = pd.read_csv(DIR_DADOS / "teste_sem_rotulo.csv")
    ids = df[COL_ID].copy()
    descartar = [COL_ID] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    return ids, df.drop(columns=descartar)


X_treino, y_treino = carregar_treino()
print(f"Treino: {X_treino.shape[0]:,} linhas x {X_treino.shape[1]} atributos candidatos "
      f"(sinais operacionais excluídos: {SINAIS_OPERACIONAIS})")
print("Distribuição do alvo:", y_treino.map({0: "normal", 1: "anomaly"}).value_counts().to_dict())

Treino: 69,833 linhas x 46 atributos candidatos (sinais operacionais excluídos: ['score_ids_legado', 'tipo_alerta_soc', 'tempo_resposta_analista_min'])
Distribuição do alvo: {'normal': 65754, 'anomaly': 4079}


### 1.2 Evidências de redundância e inconsistência

Antes de comparar estratégias, confirmamos com igualdades exatas (não com correlação) o que a etapa 1 apontou: colunas derivadas, constantes, variações de escrita e valores fora do domínio. São verificações estruturais, sem nenhum parâmetro aprendido para o modelo.

In [2]:
# Samuel Lucas Rodrigues Vieira — 1.2 Evidências estruturais: redundâncias, constantes, categorias, domínio
# Verificações determinísticas (igualdades e contagens). Não geram parâmetros usados pelo modelo.
def _rel(nome, original, derivada):
    ambos = original.notna() & derivada.notna()
    return {
        "derivada": nome,
        "relacao_testada": COLS_DERIVADAS[nome][1],
        "linhas_comparadas": int(ambos.sum()),
        "max_diferenca_abs": float((original[ambos] - derivada[ambos]).abs().max()),
        "mesmo_padrao_ausencia": bool(original.isna().equals(derivada.isna())),
        "ausentes_recuperaveis": int((original.isna() & derivada.notna()).sum()),
    }

Xe = X_treino
evid_redund = pd.DataFrame([
    _rel("src_bytes_kb", Xe["src_bytes"] / 1024, Xe["src_bytes_kb"]),
    _rel("dst_bytes_kb", Xe["dst_bytes"] / 1024, Xe["dst_bytes_kb"]),
    _rel("total_bytes", Xe["src_bytes"] + Xe["dst_bytes"], Xe["total_bytes"]),
    _rel("same_srv_pct", Xe["same_srv_rate"] * 100, Xe["same_srv_pct"]),
]).set_index("derivada")
# protocolo_cod: cada protocolo (normalizado) deve mapear para exatamente um código.
tab_proto = pd.crosstab(Xe["protocol_type"].str.strip().str.lower(), Xe["protocolo_cod"])
evid_redund.loc["protocolo_cod"] = {
    "relacao_testada": COLS_DERIVADAS["protocolo_cod"][1],
    "linhas_comparadas": int(Xe["protocol_type"].notna().sum()),
    "max_diferenca_abs": float(((tab_proto > 0).sum(axis=1) != 1).sum()),  # protocolos com >1 código
    "mesmo_padrao_ausencia": bool(Xe["protocol_type"].isna().equals(Xe["protocolo_cod"].isna())),
    "ausentes_recuperaveis": int((Xe["protocol_type"].isna() & Xe["protocolo_cod"].notna()).sum()),
}
print("Relações determinísticas (max_diferenca_abs = 0 => derivada redundante):")
display(evid_redund)
print("protocol_type normalizado x protocolo_cod:")
display(tab_proto)

constantes = [c for c in Xe.columns if Xe[c].nunique(dropna=True) <= 1]
print("Atributos constantes no treino completo:", constantes)

evid_cat = pd.DataFrame({
    "categorias_brutas": Xe[COLS_CATEGORICAS].nunique(),
    "categorias_normalizadas": [Xe[c].str.strip().str.lower().nunique() for c in COLS_CATEGORICAS],
    "ausentes": Xe[COLS_CATEGORICAS].isna().sum(),
})
print("Inconsistência de escrita nos categóricos (caixa/espaços):")
display(evid_cat)

fora = pd.DataFrame({c: (Xe[c] < lo) | (Xe[c] > hi) for c, (lo, hi) in LIMITES_DOMINIO.items()})
print(f"Células fora do domínio: {int(fora.values.sum()):,}; linhas afetadas: {int(fora.any(axis=1).sum()):,}")

Relações determinísticas (max_diferenca_abs = 0 => derivada redundante):


,relacao_testada,linhas_comparadas,max_diferenca_abs,mesmo_padrao_ausencia,ausentes_recuperaveis
derivada,,,,,
src_bytes_kb,src_bytes / 1024,41927,0.000000e+00,True,0
dst_bytes_kb,dst_bytes / 1024,69833,0.000000e+00,True,0
total_bytes,src_bytes + dst_bytes,41927,0.000000e+00,True,0
same_srv_pct,same_srv_rate * 100,66262,3.552714e-15,True,0
protocolo_cod,código 1:1 de protocol_type,69833,0.000000e+00,True,0


protocol_type normalizado x protocolo_cod:


protocolo_cod,0,1,2
protocol_type,,,
icmp,0,0,1643
tcp,56010,0,0
udp,0,12180,0


Atributos constantes no treino completo: ['num_outbound_cmds']
Inconsistência de escrita nos categóricos (caixa/espaços):


,categorias_brutas,categorias_normalizadas,ausentes
protocol_type,6,3,0
service,96,67,4416
flag,19,11,4093


Células fora do domínio: 6,121; linhas afetadas: 5,650


### 1.3 Transformações reutilizáveis

Cada etapa é um transformador do scikit-learn, encaixado em um `Pipeline`. Assim, medianas, modas, categorias do One-Hot e constantes são aprendidas **apenas no `fit`**, sempre chamado com o treino da dobra.

| Transformação | Aprende dos dados? | Opções comparadas |
|---|---|---|
| `NormalizarCategorias` | Não (strip + minúsculas) | brutas × normalizadas |
| `TratarForaDominio` | Não (limites da documentação) | manter × virar ausente × recortar |
| `RemoverRedundantes` | Sim: constantes da dobra | manter × remover derivadas + constantes |
| `SimpleImputer` numérico | Sim: mediana da dobra | mediana × mediana + indicador de ausência |
| `SimpleImputer` categórico | Sim: moda da dobra (se usada) | categoria `__ausente__` × moda |
| `filtrar_linhas_treino` | Sim: grupos do treino da dobra | todas × sem duplicatas × sem conflitos × ambos |

O One-Hot com `handle_unknown="ignore"` é só o mínimo para viabilizar o treinamento do modelo. A comparação de codificações e escalas fica com o Carlos Eduardo de Melo Sabino.

In [3]:
# Samuel Lucas Rodrigues Vieira — 1.3 Transformações reutilizáveis (compatíveis com Pipeline do scikit-learn)
# Regra: tudo que aprende algo dos dados (constantes, medianas, modas, categorias) só aprende no fit,
# que é chamado apenas com o treino de cada dobra. O teste oficial só passa por transform.

class NormalizarCategorias(BaseEstimator, TransformerMixin):
    """Remove espaços e uniformiza caixa ('TCP ' -> 'tcp'). Sem estado; NaN continua NaN."""

    def __init__(self, colunas=tuple(COLS_CATEGORICAS), ativo=True):
        self.colunas = colunas
        self.ativo = ativo

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.colunas:
            if c in X:
                serie = X[c].astype(object)
                X[c] = serie.str.strip().str.lower() if self.ativo else serie
        return X


class TratarForaDominio(BaseEstimator, TransformerMixin):
    """Valores impossíveis pelos limites documentados.

    modo='manter' (referência), 'ausente' (vira NaN e segue para a imputação)
    ou 'recorte' (clip nos limites). Os limites vêm da documentação, não dos dados.
    """

    def __init__(self, limites=None, modo="manter"):
        self.limites = limites
        self.modo = modo

    def fit(self, X, y=None):
        assert self.modo in {"manter", "ausente", "recorte"}
        return self

    def transform(self, X):
        if self.modo == "manter":
            return X
        X = X.copy()
        for c, (lo, hi) in (self.limites or LIMITES_DOMINIO).items():
            if c not in X:
                continue
            if self.modo == "ausente":
                X[c] = X[c].astype(float).mask((X[c] < lo) | (X[c] > hi))
            else:
                X[c] = X[c].clip(lower=lo, upper=hi)
        return X


class RemoverRedundantes(BaseEstimator, TransformerMixin):
    """Remove derivadas determinísticas (lista fixa) e constantes detectadas NO TREINO DA DOBRA."""

    def __init__(self, derivadas=tuple(COLS_DERIVADAS), ativo=True):
        self.derivadas = derivadas
        self.ativo = ativo

    def fit(self, X, y=None):
        if not self.ativo:
            self.remover_ = []
            return self
        constantes = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
        self.constantes_ = constantes
        self.remover_ = sorted(set(self.derivadas).intersection(X.columns) | set(constantes))
        return self

    def transform(self, X):
        return X.drop(columns=self.remover_)


def _eh_categorica(X):
    return [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]


def _eh_numerica(X):
    return [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]


def montar_preprocessamento(normalizar_categorias=True, fora_dominio="manter",
                            remover_redundantes=False, imputacao_num="mediana",
                            imputacao_cat="constante"):
    """Bloco de pré-processamento do Samuel Lucas Rodrigues Vieira (+ One-Hot mínimo, provisório até o Carlos Eduardo de Melo Sabino).

    imputacao_num: 'mediana' | 'mediana+indicador'
    imputacao_cat: 'constante' (ausência vira a categoria '__ausente__') | 'moda'
    """
    imp_num = SimpleImputer(strategy="median",
                            add_indicator=(imputacao_num == "mediana+indicador"))
    imp_cat = (SimpleImputer(strategy="constant", fill_value="__ausente__")
               if imputacao_cat == "constante" else SimpleImputer(strategy="most_frequent"))
    colunas = ColumnTransformer([
        ("num", imp_num, _eh_numerica),
        ("cat", Pipeline([("imp", imp_cat),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         _eh_categorica),
    ], verbose_feature_names_out=False)
    return Pipeline([
        ("categorias", NormalizarCategorias(ativo=normalizar_categorias)),
        ("dominio", TratarForaDominio(modo=fora_dominio)),
        ("redundantes", RemoverRedundantes(ativo=remover_redundantes)),
        ("imputacao_codificacao", colunas),
    ])


def chave_registro(X):
    """Identificador de registros com atributos idênticos (categorias normalizadas, NaN como valor)."""
    Xn = NormalizarCategorias().transform(X)
    return pd.Series(pd.factorize(pd.util.hash_pandas_object(Xn, index=False))[0], index=X.index)


def filtrar_linhas_treino(X, y, estrategia="todas"):
    """Duplicatas/conflitos tratados SOMENTE no treino da dobra; a validação fica intacta.

    'todas'                      : mantém tudo (referência)
    'sem_duplicatas'             : remove cópias com mesmos atributos e mesma classe (mantém a 1ª)
    'sem_conflitos'              : remove TODOS os registros de grupos com atributos iguais e
                                   classes divergentes (não inventa nem troca rótulos)
    'sem_duplicatas_e_conflitos' : as duas remoções
    """
    if estrategia == "todas":
        return X, y
    if estrategia not in {"sem_duplicatas", "sem_conflitos", "sem_duplicatas_e_conflitos"}:
        raise ValueError(estrategia)
    chave = chave_registro(X)
    manter = pd.Series(True, index=X.index)
    if "duplicatas" in estrategia:
        manter &= ~pd.DataFrame({"k": chave, "y": y}).duplicated()
    if "conflitos" in estrategia:
        manter &= ~chave.map(y.groupby(chave).nunique()).gt(1)
    return X[manter], y[manter]

### 1.4 Dobras sem contaminação por duplicatas

Com `StratifiedKFold` comum, cópias idênticas de um registro podem aparecer tanto no treino quanto na validação. Essa sobreposição pode favorecer a memorização e tornar a estimativa de desempenho otimista. Usamos `StratifiedGroupKFold`, com grupo = registro de atributos idênticos (categorias normalizadas, `NaN` tratado como valor). A auditoria abaixo compara as duas divisões. `INDICES_DOBRAS` fica disponível para os demais membros usarem exatamente as mesmas dobras.

In [4]:
# Samuel Lucas Rodrigues Vieira — 1.4 Dobras sem contaminação por cópias
# Registros com atributos idênticos recebem o mesmo grupo e ficam sempre na mesma dobra.
grupos_treino = chave_registro(X_treino)


def auditar_dobras(divisor, nome, usa_grupos):
    linhas = []
    args = (X_treino, y_treino, grupos_treino) if usa_grupos else (X_treino, y_treino)
    for i, (itr, iva) in enumerate(divisor.split(*args), start=1):
        chaves_tr = set(grupos_treino.iloc[itr])
        copia_no_treino = grupos_treino.iloc[iva].isin(chaves_tr)
        linhas.append({"divisao": nome, "dobra": i, "n_validacao": len(iva),
                       "pct_anomaly_validacao": 100 * y_treino.iloc[iva].mean(),
                       "pct_validacao_com_copia_no_treino": 100 * copia_no_treino.mean()})
    return linhas


DOBRAS = StratifiedGroupKFold(n_splits=N_DOBRAS, shuffle=True, random_state=SEMENTE)
auditoria = pd.DataFrame(
    auditar_dobras(StratifiedKFold(N_DOBRAS, shuffle=True, random_state=SEMENTE), "StratifiedKFold", False)
    + auditar_dobras(DOBRAS, "StratifiedGroupKFold (grupos = registro)", True)
)
print(f"Grupos de registros distintos: {grupos_treino.nunique():,} para {len(grupos_treino):,} linhas")
display(auditoria.groupby("divisao")[["n_validacao", "pct_anomaly_validacao",
                                      "pct_validacao_com_copia_no_treino"]].agg(["mean", "min", "max"]))
# Estrutura dos conflitos (atributos idênticos, classes diferentes) no treino completo — só diagnóstico.
perfil_grupos = y_treino.groupby(grupos_treino).agg(["size", "sum", "nunique"])
conflitos = perfil_grupos[perfil_grupos["nunique"] > 1]
print(f"Grupos conflitantes: {len(conflitos):,}; linhas: {int(conflitos['size'].sum()):,}; "
      f"anomalias envolvidas: {int(conflitos['sum'].sum()):,} de {int(y_treino.sum()):,} "
      f"({100 * conflitos['sum'].sum() / y_treino.sum():.1f}%)")
print("Composição dos grupos conflitantes (linhas x anomalias no grupo):")
display(pd.crosstab(conflitos["size"].rename("linhas_no_grupo"), conflitos["sum"].rename("anomalias_no_grupo")))

# Índices fixos das dobras, para os demais membros reutilizarem exatamente as mesmas divisões.
INDICES_DOBRAS = list(DOBRAS.split(X_treino, y_treino, grupos_treino))

Grupos de registros distintos: 56,766 para 69,833 linhas


n_validacao                \
                                                mean    min    max   
divisao                                                              
StratifiedGroupKFold (grupos = registro)     13966.6  13966  13968   
StratifiedKFold                              13966.6  13966  13967   

                                         pct_anomaly_validacao            \
                                                          mean       min   
divisao                                                                    
StratifiedGroupKFold (grupos = registro)              5.841078  5.835601   
StratifiedKFold                                       5.841078  5.835601   

                                                    \
                                               max   
divisao                                              
StratifiedGroupKFold (grupos = registro)  5.842761   
StratifiedKFold                           5.842761   

                                         pct_validacao_com_copia_no_treino  \
                                                                      mean   
divisao                                                                      
StratifiedGroupKFold (grupos = registro)                          0.000000   
StratifiedKFold                                                  29.709452   

                                                                
                                                min        max  
divisao                                                         
StratifiedGroupKFold (grupos = registro)   0.000000   0.000000  
StratifiedKFold                           29.505262  29.956326

Grupos conflitantes: 1,702; linhas: 3,730; anomalias envolvidas: 1,896 de 4,079 (46.5%)
Composição dos grupos conflitantes (linhas x anomalias no grupo):


anomalias_no_grupo,1,2
linhas_no_grupo,,
2,1376,0
3,132,194


### 1.5 Comparação com a referência (um fator por vez)

- **R0 (referência):** categorias brutas, valores fora do domínio mantidos, mediana nos numéricos, categoria `__ausente__` nos categóricos, todas as colunas e todas as linhas.
- Cada variação altera **um único fator** em relação a R0.
- Modelo provisório: `DecisionTreeClassifier(random_state=42)`. Árvores não dependem de escala, o que isola os efeitos desta seção do trabalho do Carlos Eduardo de Melo Sabino.
- Métrica principal: F1-macro na validação de cada dobra. A validação **nunca** é filtrada (duplicatas e conflitos continuam nela), então a estimativa reflete dados com o mesmo ruído.

In [5]:
# Samuel Lucas Rodrigues Vieira — 1.5 Comparação das estratégias (um fator por vez contra a referência)
# Modelo de referência provisório: árvore de decisão (insensível a escala, isola os efeitos do Samuel Lucas Rodrigues Vieira).
# Mesmo modelo, mesma semente e mesmas dobras para todas as estratégias.
MODELO_REFERENCIA = DecisionTreeClassifier(random_state=SEMENTE)

REFERENCIA = dict(normalizar_categorias=False, fora_dominio="manter", remover_redundantes=False,
                  imputacao_num="mediana", imputacao_cat="constante", linhas_treino="todas")
VARIACOES = {
    "R0 referência": ("—", {}),
    "A1 categorias normalizadas": ("categorias", {"normalizar_categorias": True}),
    "B1 categórico: moda": ("ausentes categóricos", {"imputacao_cat": "moda"}),
    "B2 numérico: mediana + indicador": ("ausentes numéricos", {"imputacao_num": "mediana+indicador"}),
    "C1 fora do domínio -> ausente": ("inconsistências de domínio", {"fora_dominio": "ausente"}),
    "C2 fora do domínio -> recorte": ("inconsistências de domínio", {"fora_dominio": "recorte"}),
    "D1 remover derivadas e constantes": ("redundância de atributos", {"remover_redundantes": True}),
    "E1 treino sem duplicatas": ("redundância de registros", {"linhas_treino": "sem_duplicatas"}),
    "E2 treino sem conflitos": ("redundância de registros", {"linhas_treino": "sem_conflitos"}),
    "E3 treino sem duplicatas e conflitos": ("redundância de registros",
                                             {"linhas_treino": "sem_duplicatas_e_conflitos"}),
}


def avaliar_estrategia(config, modelo=MODELO_REFERENCIA, dobras=INDICES_DOBRAS):
    """Ajusta pré-processamento + modelo só no treino de cada dobra e avalia na validação intacta."""
    config = dict(config)
    linhas_treino = config.pop("linhas_treino")
    resultados = []
    for i, (itr, iva) in enumerate(dobras, start=1):
        X_tr, y_tr = filtrar_linhas_treino(X_treino.iloc[itr], y_treino.iloc[itr], linhas_treino)
        X_va, y_va = X_treino.iloc[iva], y_treino.iloc[iva]
        pipe = Pipeline([("prep", montar_preprocessamento(**config)), ("modelo", clone(modelo))])
        t0 = time.perf_counter()
        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_va)
        f1_normal, f1_anomaly = f1_score(y_va, pred, average=None, labels=[0, 1])
        resultados.append({
            "dobra": i, "f1_macro": (f1_normal + f1_anomaly) / 2,
            "f1_normal": f1_normal, "f1_anomaly": f1_anomaly,
            "n_linhas_treino": len(X_tr), "n_atributos_finais": pipe[:-1].transform(X_va.head(1)).shape[1],
            "segundos": time.perf_counter() - t0,
        })
    return pd.DataFrame(resultados)


por_dobra = []
for nome, (fator, mudanca) in VARIACOES.items():
    res = avaliar_estrategia({**REFERENCIA, **mudanca})
    res.insert(0, "estrategia", nome)
    res.insert(1, "fator", fator)
    por_dobra.append(res)
por_dobra_m1 = pd.concat(por_dobra, ignore_index=True)

ref_f1 = por_dobra_m1.loc[por_dobra_m1.estrategia == "R0 referência"].set_index("dobra")["f1_macro"]
por_dobra_m1["delta_vs_R0"] = por_dobra_m1["f1_macro"] - por_dobra_m1["dobra"].map(ref_f1)

tabela_m1 = por_dobra_m1.groupby(["estrategia", "fator"], sort=False).agg(
    f1_macro_media=("f1_macro", "mean"), f1_macro_dp=("f1_macro", "std"),
    f1_anomaly=("f1_anomaly", "mean"), f1_normal=("f1_normal", "mean"),
    delta_vs_R0=("delta_vs_R0", "mean"),
    dobras_melhores_que_R0=("delta_vs_R0", lambda d: int((d > 0).sum())),
    linhas_treino=("n_linhas_treino", "mean"), atributos=("n_atributos_finais", "mean"),
    segundos=("segundos", "mean"),
).reset_index()
with pd.option_context("display.float_format", "{:.4f}".format):
    display(tabela_m1)
    print("F1-macro por dobra:")
    display(por_dobra_m1.pivot(index="estrategia", columns="dobra", values="f1_macro")
            .reindex(VARIACOES))

,estrategia,fator,f1_macro_media,f1_macro_dp,f1_anomaly,f1_normal,delta_vs_R0,dobras_melhores_que_R0,linhas_treino,atributos,segundos
0,R0 referência,—,0.8049,0.0054,0.6297,0.9800,0.0000,0,55866.4000,162.0000,2.0573
1,A1 categorias normalizadas,categorias,0.8027,0.0072,0.6255,0.9799,-0.0021,1,55866.4000,125.2000,1.7143
2,B1 categórico: moda,ausentes categóricos,0.8055,0.0082,0.6311,0.9800,0.0007,2,55866.4000,160.0000,2.2125
3,B2 numérico: mediana + indicador,ausentes numéricos,0.8048,0.0035,0.6296,0.9800,-0.0001,2,55866.4000,187.0000,2.3059
4,C1 fora do domínio -> ausente,inconsistências de domínio,0.8051,0.0043,0.6302,0.9800,0.0003,3,55866.4000,162.0000,2.3018
5,C2 fora do domínio -> recorte,inconsistências de domínio,0.8040,0.0063,0.6280,0.9799,-0.0009,1,55866.4000,162.0000,2.0437
6,D1 remover derivadas e constantes,redundância de atributos,0.8055,0.0056,0.6310,0.9800,0.0006,2,55866.4000,155.8000,1.9908
7,E1 treino sem duplicatas,redundância de registros,0.7817,0.0094,0.5847,0.9787,-0.0232,0,46774.4000,162.0000,2.2971
8,E2 treino sem conflitos,redundância de registros,0.8717,0.0041,0.7587,0.9847,0.0668,5,52882.4000,157.4000,1.7315
9,E3 treino sem duplicatas e conflitos,redundância de registros,0.8698,0.0046,0.7552,0.9845,0.0650,5,44051.2000,157.4000,1.3148


F1-macro por dobra:


dobra,1,2,3,4,5
estrategia,,,,,
R0 referência,0.8059,0.8106,0.8084,0.8028,0.7966
A1 categorias normalizadas,0.7963,0.8036,0.8074,0.8116,0.7948
B1 categórico: moda,0.7965,0.8092,0.8179,0.8010,0.8032
B2 numérico: mediana + indicador,0.8012,0.8089,0.8022,0.8036,0.8082
C1 fora do domínio -> ausente,0.8123,0.8030,0.8009,0.8050,0.8045
C2 fora do domínio -> recorte,0.8017,0.8044,0.8051,0.8131,0.7957
D1 remover derivadas e constantes,0.7957,0.8099,0.8069,0.8076,0.8073
E1 treino sem duplicatas,0.7662,0.7839,0.7860,0.7811,0.7912
E2 treino sem conflitos,0.8702,0.8724,0.8784,0.8681,0.8693


### 1.6 Combinação proposta e compatibilidade com o teste

Combinamos as opções justificadas na análise desta seção e verificamos que o pipeline ajustado **somente no treino** transforma o teste oficial sem remover nem reordenar linhas e sem deixar ausentes. Nenhuma estatística do teste é calculada e nenhuma previsão é gerada.

In [6]:
# Samuel Lucas Rodrigues Vieira — 1.6 Combinação proposta e checagem de compatibilidade com o teste oficial
# Opções escolhidas pelos resultados da 1.5 e por justificativa técnica (ver tabela 1.7).
# Atenção: escolher e reavaliar nas mesmas dobras é otimista; o Paulo Gabriel de Oliveira Leite deve confirmar na integração.
CONFIG_M1 = dict(normalizar_categorias=True, fora_dominio="ausente", remover_redundantes=True,
                 imputacao_num="mediana", imputacao_cat="constante", linhas_treino="sem_conflitos")

res_comb = avaliar_estrategia(CONFIG_M1)
res_comb.insert(0, "estrategia", "F1 combinação proposta")
res_comb["delta_vs_R0"] = res_comb["f1_macro"] - res_comb["dobra"].map(ref_f1)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(res_comb)
print(f"F1-macro combinação: {res_comb.f1_macro.mean():.4f} ± {res_comb.f1_macro.std():.4f} "
      f"(R0: {ref_f1.mean():.4f}); melhor que R0 em {(res_comb.delta_vs_R0 > 0).sum()}/{N_DOBRAS} dobras")

# Compatibilidade: ajusta SÓ no treino e transforma o teste sem remover/reordenar linhas.
# Nenhuma estatística do teste é usada; nenhuma previsão é gerada aqui.
config_prep = {k: v for k, v in CONFIG_M1.items() if k != "linhas_treino"}
X_fit, y_fit = filtrar_linhas_treino(X_treino, y_treino, CONFIG_M1["linhas_treino"])
prep_m1 = montar_preprocessamento(**config_prep).fit(X_fit, y_fit)
ids_teste, X_teste = carregar_teste()
assert list(X_teste.columns) == list(X_treino.columns), "Esquema do teste difere do treino."
Z_teste = prep_m1.transform(X_teste)
assert Z_teste.shape[0] == len(ids_teste) and not np.isnan(Z_teste).any()
print(f"Teste transformado: {Z_teste.shape[0]:,} linhas (ordem e IDs preservados), "
      f"{Z_teste.shape[1]} colunas, sem ausentes.")
print("Colunas removidas no ajuste:", prep_m1.named_steps["redundantes"].remover_)

,estrategia,dobra,f1_macro,f1_normal,f1_anomaly,n_linhas_treino,n_atributos_finais,segundos,delta_vs_R0
0,F1 combinação proposta,1,0.8685,0.9841,0.7528,52884,117,1.2562,0.0626
1,F1 combinação proposta,2,0.8703,0.9845,0.7561,52879,117,1.2805,0.0597
2,F1 combinação proposta,3,0.8779,0.9853,0.7705,52864,117,1.6289,0.0695
3,F1 combinação proposta,4,0.8664,0.9843,0.7485,52891,117,1.3792,0.0636
4,F1 combinação proposta,5,0.8701,0.9845,0.7557,52894,117,1.3394,0.0734


F1-macro combinação: 0.8706 ± 0.0044 (R0: 0.8049); melhor que R0 em 5/5 dobras
Teste transformado: 24,333 linhas (ordem e IDs preservados), 118 colunas, sem ausentes.
Colunas removidas no ajuste: ['dst_bytes_kb', 'num_outbound_cmds', 'protocolo_cod', 'same_srv_pct', 'src_bytes_kb', 'total_bytes']


### Resultados e justificativa — Samuel Lucas Rodrigues Vieira

**Evidências estruturais (1.2).** As cinco colunas derivadas reproduzem exatamente a coluna de origem: `src_bytes_kb` = `src_bytes`/1024, `dst_bytes_kb` = `dst_bytes`/1024, `total_bytes` = `src_bytes` + `dst_bytes`, `same_srv_pct` = 100 × `same_srv_rate` (diferença máxima 4e-15, erro de ponto flutuante) e `protocolo_cod` 1:1 com `protocol_type` normalizado. Todas têm **o mesmo padrão de ausência** da origem, então não recuperam nenhum ausente. Essas colunas não acrescentam informação aos atributos de origem nas verificações realizadas. `num_outbound_cmds` é constante. Os categóricos caem de 6→3, 96→67 e 19→11 categorias após normalizar caixa e espaços. Há 6.121 células fora do domínio, em 5.650 linhas.

**Duplicatas e conflitos (1.4).** As 69.833 linhas formam 56.766 registros distintos. Com `StratifiedKFold`, **~29,7% das linhas de validação têm uma cópia idêntica no treino da dobra**. Com `StratifiedGroupKFold` agrupado por registro, 0%, mantendo 5,84% de `anomaly` em todas as dobras. Os grupos com rótulos conflitantes merecem atenção especial. São 1.702 grupos (3.730 linhas) com atributos idênticos e classes opostas, quase todos pares 1 normal + 1 anomaly. Eles concentram **1.896 das 4.079 anomalias (46,5%)**. O padrão sugere ruído de rótulo (cópias com rótulo trocado), mas não temos como saber qual rótulo é o verdadeiro.

**Comparação (1.5), F1-macro médio em 5 dobras (R0 = 0,8049 ± 0,0054):**

| Fator | Estratégias comparadas | Evidência (Δ F1-macro vs R0, dobras melhores) | Decisão e justificativa |
|---|---|---|---|
| Categorias inconsistentes | brutas × normalizadas | −0,0021, 1/5 (diferença pequena diante da variação entre dobras, dp ≈ 0,005–0,007) | **Normalizar.** Desempenho equivalente com 162 → 125 colunas, sem isso, `TCP` e `tcp` seriam categorias diferentes, e uma variação nova no teste viraria "desconhecida". |
| Ausentes categóricos | `__ausente__` × moda | +0,0007, 2/5 (empate) | **Categoria `__ausente__`.** Empate no desempenho, mantém a ausência visível em vez de inventar o valor mais comum. |
| Ausentes numéricos | mediana × mediana + indicador | −0,0001, 2/5 (empate) | **Mediana sem indicador** para a árvore (+25 colunas sem ganho). Como `dst_host_serror_rate` tem ausência informativa (etapa 1), reavaliar o indicador com modelos lineares/MLP. |
| Valores fora do domínio | manter × ausente × recorte | ausente +0,0003 (3/5), recorte −0,0009 (1/5) | **Tratar como ausente.** Pequena vantagem e coerência: um valor impossível é erro de registro, e recortar inventaria um valor no limite. |
| Redundância de atributos | manter × remover derivadas + constantes | +0,0006, 2/5 | **Remover.** Igualdade exata comprovada, nenhum ausente recuperável, reduz 6 colunas sem perda. Constantes são detectadas no treino de cada dobra. |
| Duplicatas no treino | manter × remover | −0,0232, 0/5 | **Manter.** Remover cópias muda o peso das amostras e piora todas as dobras, a contaminação já é evitada pelo agrupamento das dobras. |
| Conflitos no treino | manter × remover grupos conflitantes | **+0,0668, 5/5** (0,8717 ± 0,0041, F1 anomaly 0,630 → 0,759) | **Remover só do treino de cada dobra**, sem trocar rótulos. A validação continua com os conflitos, então o ganho não vem de facilitar a avaliação. Remover também as duplicatas (E3) não acrescenta nada (0,8698). |

**Combinação proposta (1.6):** categorias normalizadas, fora do domínio → ausente, remoção de derivadas e constantes, mediana + `__ausente__` e treino sem conflitos. Resultado: **F1-macro 0,8706 ± 0,0044**, melhor que R0 nas 5 dobras, com 117 colunas (R0: 162). O ganho vem quase todo dos conflitos. Os demais fatores são neutros em desempenho e foram escolhidos por coerência e menor dimensão. O pipeline ajustado só no treino transforma as 24.333 linhas do teste sem perdas, sem reordenação e sem ausentes.

**Limitações e pendências para a integração (Paulo Gabriel de Oliveira Leite).**
- As opções foram escolhidas e reavaliadas nas mesmas dobras, o que torna a estimativa da combinação um pouco otimista. Confirmar com outra semente de dobras e com os modelos da etapa 3.
- O efeito dos conflitos foi medido com uma árvore. Conferir se persiste com os demais modelos e com o balanceamento do Matheus de Almeida Moreira.
- O agrupamento considera iguais apenas registros idênticos, inclusive nos `NaN`. Cópias que diferem só por um valor ausente não são agrupadas.
- Os sinais operacionais permanecem excluídos na integração da parte 5, enquanto sua origem e disponibilidade temporal não forem confirmadas.
- Nenhuma decisão usou o teste. O teste oficial não foi alterado.

## Matheus de Almeida Moreira — Duas técnicas de balanceamento

**Objetivo da seção:** Escolher e implementar pelo menos duas técnicas, comparando-as com nenhum balanceamento nas mesmas dobras e com o mesmo modelo de referência. Exemplos: SMOTE versus RandomUnderSampler, ou RandomOverSampler versus RandomUnderSampler. Justificar o tratamento de categorias caso use método sintético.

**Conteúdo apresentado:** Comparação de F1-macro e métricas por classe, distribuição antes/depois apenas no treino de cada dobra e discussão de perda de informação/sobreajuste.


### 2.1 Protocolo e escolha dos balanceamentos

Comparamos **sem balanceamento**, **RandomOverSampler (ROS)**, **RandomUnderSampler (RUS)** e **SMOTENC + Tomek Links**, usando `INDICES_DOBRAS` e a configuração `CONFIG_M1` (inclusive remoção de conflitos somente no treino). A árvore da parte 1 continua como referência. O **XGBoost** é um experimento adicional, com os mesmos balanceamentos e divisões. Assim, comparamos balanceamentos **dentro de cada modelo**, sem atribuir ao balanceamento uma melhora que pode vir do classificador.

A proposta SMOTETomek foi adaptada para dados mistos: **SMOTENC → One-Hot → TomekLinks**. Não usamos a classe `SMOTETomek` padrão: SMOTE sobre One-Hot pode interpolar indicadores e produzir categorias fracionárias. SMOTENC escolhe categorias existentes. O One-Hot posterior permite calcular os vizinhos de Tomek sem impor uma ordem artificial às categorias. Os indicadores discretos conhecidos também são tratados como categorias. Contagens numéricas ainda podem ficar fracionárias nos exemplos sintéticos: eles são vetores de treinamento, não conexões reais reconstruídas.

Imputação e `StandardScaler` são ajustados **antes da reamostragem, só no treino da dobra**. A escala é necessária para as distâncias de SMOTENC/Tomek e fica igual em todas as estratégias. Isto não substitui a comparação de escalas do Carlos Eduardo de Melo Sabino. ROS e SMOTENC visam proporção 1:1. RUS reduz a maioria até 1:1, sem reposição. Tomek remove os dois lados dos pares (`sampling_strategy="all"`), reduzindo o total de linhas após a reamostragem. Esses parâmetros são fixados antes da comparação, sem busca nas dobras.

O XGBoost usa 200 árvores, profundidade 4, taxa 0,05 e limiar padrão 0,5. `scale_pos_weight=1` evita misturar ponderação com reamostragem. Não há early stopping orientado pela validação nem ajuste de hiperparâmetros. Validação e teste oficial nunca são balanceados. Esta seção nem lê o teste. Os sinais operacionais continuam excluídos.

Fontes: [SMOTENC](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html), [SMOTETomek](https://imbalanced-learn.org/stable/references/generated/imblearn.combine.SMOTETomek.html), [TomekLinks](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.TomekLinks.html) e [XGBClassifier](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier).

In [7]:
# Matheus de Almeida Moreira — blocos reutilizáveis, com reamostragem apenas durante fit.
import platform
import sklearn
import imblearn
import xgboost
from imblearn.over_sampling import RandomOverSampler, SMOTENC
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.pipeline import Pipeline as PipelineBalanceado
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits

print({"Python": platform.python_version(), "numpy": np.__version__,
       "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
       "imbalanced-learn": imblearn.__version__, "xgboost": xgboost.__version__})

COLS_NOMINAIS_M2 = tuple(COLS_CATEGORICAS) + (
    "land", "logged_in", "root_shell", "su_attempted", "is_host_login", "is_guest_login",
)
ESTRATEGIAS_M2 = ("Sem balanceamento", "ROS", "RUS", "SMOTENC + Tomek")
MODELOS_M2 = {
    "Árvore": clone(MODELO_REFERENCIA),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=1.0, colsample_bytree=1.0, scale_pos_weight=1,
        objective="binary:logistic", eval_metric="logloss", tree_method="hist",
        random_state=SEMENTE, n_jobs=2,
    ),
}


def contagens_m2(y):
    return pd.Series(np.asarray(y)).value_counts().reindex([0, 1], fill_value=0).astype(int).to_dict()


def nominais_m2(X):
    return [c for c in COLS_NOMINAIS_M2 if c in X.columns]


def numericas_m2(X):
    return [c for c in X.columns if c not in COLS_NOMINAIS_M2]


class CategoriasM2(BaseEstimator, TransformerMixin):
    """Converte indicadores discretos para texto, preservando NaN para a imputação."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in nominais_m2(X):
            X[c] = X[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
        return X


class AmostradorAuditadoM2(BaseEstimator):
    """Registra contagens antes/depois; não participa de predict/transform."""
    def __init__(self, amostrador=None):
        self.amostrador = amostrador

    def fit_resample(self, X, y):
        self.antes_ = contagens_m2(y)
        self.amostrador_ = clone(self.amostrador) if self.amostrador is not None else None
        Xr, yr = (X, y) if self.amostrador_ is None else self.amostrador_.fit_resample(X, y)
        self.depois_ = contagens_m2(yr)
        if isinstance(self.amostrador_, SMOTENC):
            # SMOTENC deve preservar os domínios categóricos aprendidos nesta dobra.
            for c in nominais_m2(X):
                assert set(Xr[c].unique()).issubset(set(X[c].unique())), c
        return Xr, yr


def montar_pipeline_m2(estrategia, modelo, config=None):
    """Pipeline reutilizável. O chamador filtra conflitos apenas no treino antes de fit.

    Todas as estatísticas são ajustadas no fit; os amostradores são pulados em predict.
    """
    if estrategia not in ESTRATEGIAS_M2:
        raise ValueError(estrategia)
    cfg = dict(CONFIG_M1 if config is None else config)
    if cfg["imputacao_num"] != "mediana" or cfg["imputacao_cat"] != "constante":
        raise ValueError("Este bloco usa mediana e categoria de ausência, conforme CONFIG_M1.")
    numericos = Pipeline([
        ("imputar", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("escala", StandardScaler()),
    ])
    misto = ColumnTransformer([
        ("num", numericos, numericas_m2),
        ("cat", SimpleImputer(strategy="constant", fill_value="__ausente__",
                              keep_empty_features=True), nominais_m2),
    ], verbose_feature_names_out=False).set_output(transform="pandas")
    # Os nomes sobrevivem à imputação; constantes podem ter sido removidas no fit.
    # SMOTENC recebe a máscara categórica por dtype, sem índices fixos frágeis.
    amostradores = {
        "Sem balanceamento": None,
        "ROS": RandomOverSampler(sampling_strategy=1.0, random_state=SEMENTE),
        "RUS": RandomUnderSampler(sampling_strategy=1.0, replacement=False, random_state=SEMENTE),
        "SMOTENC + Tomek": SMOTENC(categorical_features="auto", sampling_strategy=1.0,
                                   k_neighbors=5, random_state=SEMENTE),
    }
    codificar = ColumnTransformer([
        ("num", "passthrough", numericas_m2),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominais_m2),
    ])
    return PipelineBalanceado([
        ("categorias", NormalizarCategorias(ativo=cfg["normalizar_categorias"])),
        ("dominio", TratarForaDominio(modo=cfg["fora_dominio"])),
        ("redundantes", RemoverRedundantes(ativo=cfg["remover_redundantes"])),
        ("tipos", CategoriasM2()),
        ("imputar_escalar", misto),
        ("categorias_smote", DtypeCategoriasM2()),
        ("reamostrar", AmostradorAuditadoM2(amostradores[estrategia])),
        ("codificar", codificar),
        ("tomek", AmostradorAuditadoM2(
            TomekLinks(sampling_strategy="all", n_jobs=2) if estrategia == "SMOTENC + Tomek" else None)),
        ("modelo", clone(modelo)),
    ])


class DtypeCategoriasM2(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in nominais_m2(X):
            X[c] = X[c].astype("category")
        return X

{'Python': '3.13.14', 'numpy': '2.5.2', 'pandas': '3.0.5', 'scikit-learn': '1.9.0', 'imbalanced-learn': '0.14.2', 'xgboost': '3.4.1'}


### 2.2 Execução nas cinco dobras comuns

Cada combinação recebe um pipeline novo. Registramos precisão, recall e F1 das duas classes, F1-macro, tempo e distribuições em quatro pontos: treino original, após filtro de conflitos, após reamostragem e após Tomek. As contagens da validação são registradas separadamente e permanecem iguais entre todas as combinações. O tempo inclui ajuste e predição e depende da máquina.

In [8]:
# Matheus de Almeida Moreira — 4 estratégias × 2 modelos × 5 dobras, sem consultar o teste oficial.
def avaliar_balanceamentos_m2(config=None, dobras=None, modelos=None):
    cfg = dict(CONFIG_M1 if config is None else config)
    dobras = INDICES_DOBRAS if dobras is None else dobras
    modelos = MODELOS_M2 if modelos is None else modelos
    resultados, distribuicoes = [], []
    for dobra, (itr, iva) in enumerate(dobras, start=1):
        assert set(grupos_treino.iloc[itr]).isdisjoint(grupos_treino.iloc[iva])
        X_original, y_original = X_treino.iloc[itr], y_treino.iloc[itr]
        X_tr, y_tr = filtrar_linhas_treino(X_original, y_original, cfg["linhas_treino"])
        X_va, y_va = X_treino.iloc[iva].copy(), y_treino.iloc[iva].copy()
        for nome_modelo, modelo in modelos.items():
            for estrategia in ESTRATEGIAS_M2:
                pipe = montar_pipeline_m2(estrategia, modelo, cfg)
                inicio = time.perf_counter()
                # Limita threads numéricas para não saturar a máquina nos vizinhos mais próximos.
                with threadpool_limits(limits=2):
                    pipe.fit(X_tr, y_tr)
                    pred = pipe.predict(X_va)
                segundos = time.perf_counter() - inicio
                assert len(pred) == len(y_va)
                assert X_va.equals(X_treino.iloc[iva]) and y_va.equals(y_treino.iloc[iva])
                p, r, f, suporte = precision_recall_fscore_support(
                    y_va, pred, labels=[0, 1], zero_division=0)
                registro = {"modelo": nome_modelo, "estrategia": estrategia, "dobra": dobra,
                            "f1_macro": float(f.mean()), "segundos": segundos}
                for j, classe in enumerate(CLASSES):
                    registro.update({f"precisao_{classe}": p[j], f"recall_{classe}": r[j],
                                     f"f1_{classe}": f[j], f"suporte_{classe}": int(suporte[j])})
                resultados.append(registro)
                etapas = {"original": contagens_m2(y_original), "filtrado": contagens_m2(y_tr),
                          "reamostrado": pipe.named_steps["reamostrar"].depois_,
                          "final": pipe.named_steps["tomek"].depois_,
                          "validacao": contagens_m2(y_va)}
                dist = {"modelo": nome_modelo, "estrategia": estrategia, "dobra": dobra}
                for etapa, contagens in etapas.items():
                    for j, classe in enumerate(CLASSES):
                        dist[f"{etapa}_{classe}"] = contagens[j]
                distribuicoes.append(dist)
        print(f"Dobra {dobra}/{len(dobras)} concluída.", flush=True)
    return pd.DataFrame(resultados), pd.DataFrame(distribuicoes)


por_dobra_m2, distribuicoes_m2 = avaliar_balanceamentos_m2()
referencia_m2 = por_dobra_m2[por_dobra_m2.estrategia == "Sem balanceamento"]
por_dobra_m2 = por_dobra_m2.merge(
    referencia_m2[["modelo", "dobra", "f1_macro"]].rename(columns={"f1_macro": "f1_sem_balanceamento"}),
    on=["modelo", "dobra"], validate="many_to_one",
)
por_dobra_m2["delta_f1"] = por_dobra_m2.f1_macro - por_dobra_m2.f1_sem_balanceamento

# Auditoria: validação intacta, mesma distribuição entre modelos e 1:1 nos amostradores.
assert distribuicoes_m2.groupby("dobra")[["validacao_normal", "validacao_anomaly"]].nunique().eq(1).all().all()
cols_contagens_m2 = [c for c in distribuicoes_m2 if c not in ["modelo", "estrategia", "dobra"]]
assert distribuicoes_m2.groupby(["estrategia", "dobra"])[cols_contagens_m2].nunique().eq(1).all().all()
balanceadas_m2 = distribuicoes_m2[distribuicoes_m2.estrategia != "Sem balanceamento"]
assert balanceadas_m2.reamostrado_normal.eq(balanceadas_m2.reamostrado_anomaly).all()
sem_m2 = distribuicoes_m2[distribuicoes_m2.estrategia == "Sem balanceamento"]
assert sem_m2.filtrado_normal.eq(sem_m2.final_normal).all()
assert sem_m2.filtrado_anomaly.eq(sem_m2.final_anomaly).all()
assert distribuicoes_m2.final_normal.le(distribuicoes_m2.reamostrado_normal).all()
assert distribuicoes_m2.final_anomaly.le(distribuicoes_m2.reamostrado_anomaly).all()
print("Auditorias aprovadas: grupos disjuntos, categorias válidas e validação intacta.")

Dobra 1/5 concluída.
Dobra 2/5 concluída.
Dobra 3/5 concluída.
Dobra 4/5 concluída.
Dobra 5/5 concluída.
Auditorias aprovadas: grupos disjuntos, categorias válidas e validação intacta.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
invalid literal for int() with base 10: ''
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/backend/context.py", line 249, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/backend/context.py", line 318, in _count_physical_cores_darwin
    return int(cpu_info)


### 2.3 Resultados e distribuição das classes

O desvio padrão resume a variação entre as cinco dobras. Não é um intervalo de confiança. `delta_f1` compara cada estratégia com **o mesmo modelo sem balanceamento na mesma dobra**. As distribuições são idênticas para os dois modelos, por isso são exibidas uma única vez. `distribuicoes_m2` conserva todas as contagens, inclusive antes da remoção de conflitos e após a geração das amostras sintéticas.

In [9]:
metricas_m2 = [f"{metrica}_{classe}" for classe in CLASSES for metrica in ["precisao", "recall", "f1"]]
tabela_m2 = por_dobra_m2.groupby(["modelo", "estrategia"], sort=False).agg(
    f1_macro_media=("f1_macro", "mean"), f1_macro_dp=("f1_macro", "std"),
    delta_f1=("delta_f1", "mean"),
    dobras_melhores=("delta_f1", lambda s: int((s > 0).sum())),
    segundos_media=("segundos", "mean"),
    **{metrica: (metrica, "mean") for metrica in metricas_m2},
).reset_index()
with pd.option_context("display.float_format", "{:.4f}".format, "display.max_columns", None):
    display(tabela_m2[["modelo", "estrategia", "f1_macro_media", "f1_macro_dp", "delta_f1",
                       "dobras_melhores", "segundos_media"]])
    display(tabela_m2[["modelo", "estrategia"] + metricas_m2])
    display(por_dobra_m2.pivot(index=["modelo", "estrategia"], columns="dobra", values="f1_macro"))

# Uma linha por dobra/estratégia, pois os dois modelos recebem as mesmas contagens.
dist_exibir_m2 = distribuicoes_m2[distribuicoes_m2.modelo == "Árvore"].copy()
dist_exibir_m2["removidos_tomek"] = (
    dist_exibir_m2.reamostrado_normal + dist_exibir_m2.reamostrado_anomaly
    - dist_exibir_m2.final_normal - dist_exibir_m2.final_anomaly
)
with pd.option_context("display.max_rows", 30, "display.max_columns", None):
    display(dist_exibir_m2.drop(columns="modelo"))

# Candidatos exploratórios para a integração; não são configurações finais aprovadas.
candidatos_m2 = tabela_m2.loc[tabela_m2.groupby("modelo").f1_macro_media.idxmax(),
                             ["modelo", "estrategia", "f1_macro_media"]].reset_index(drop=True)
display(candidatos_m2)

,modelo,estrategia,f1_macro_media,f1_macro_dp,delta_f1,dobras_melhores,segundos_media
0,Árvore,Sem balanceamento,0.8718,0.0040,0.0000,0,0.8783
1,Árvore,ROS,0.8668,0.0033,-0.0049,0,0.8954
2,Árvore,RUS,0.7991,0.0078,-0.0726,0,0.2633
3,Árvore,SMOTENC + Tomek,0.8633,0.0041,-0.0085,0,15.5179
4,XGBoost,Sem balanceamento,0.8705,0.0042,0.0000,0,0.8294
5,XGBoost,ROS,0.8597,0.0041,-0.0108,0,1.4952
6,XGBoost,RUS,0.8424,0.0033,-0.0281,0,0.3535
7,XGBoost,SMOTENC + Tomek,0.8619,0.0025,-0.0087,0,15.9110


,modelo,estrategia,precisao_normal,recall_normal,f1_normal,precisao_anomaly,recall_anomaly,f1_anomaly
0,Árvore,Sem balanceamento,0.9863,0.9830,0.9846,0.7399,0.7791,0.7589
1,Árvore,ROS,0.9850,0.9835,0.9843,0.7409,0.7583,0.7494
2,Árvore,RUS,0.9882,0.9515,0.9695,0.5117,0.8161,0.6288
3,Árvore,SMOTENC + Tomek,0.9859,0.9809,0.9834,0.7154,0.7732,0.7431
4,XGBoost,Sem balanceamento,0.9845,0.9854,0.9850,0.7617,0.7507,0.7561
5,XGBoost,ROS,0.9881,0.9759,0.9820,0.6761,0.8112,0.7374
6,XGBoost,RUS,0.9882,0.9695,0.9788,0.6236,0.8137,0.7060
7,XGBoost,SMOTENC + Tomek,0.9869,0.9786,0.9828,0.6969,0.7911,0.7410


dobra                          1      2      3      4      5
modelo  estrategia                                          
XGBoost ROS               0.8591 0.8615 0.8589 0.8651 0.8539
        RUS               0.8455 0.8454 0.8423 0.8376 0.8412
        SMOTENC + Tomek   0.8589 0.8641 0.8648 0.8604 0.8612
        Sem balanceamento 0.8670 0.8714 0.8756 0.8657 0.8730
Árvore  ROS               0.8630 0.8687 0.8710 0.8643 0.8672
        RUS               0.8071 0.8020 0.7867 0.8029 0.7969
        SMOTENC + Tomek   0.8597 0.8693 0.8592 0.8633 0.8648
        Sem balanceamento 0.8699 0.8715 0.8784 0.8679 0.8709

,estrategia,dobra,original_normal,original_anomaly,filtrado_normal,filtrado_anomaly,reamostrado_normal,reamostrado_anomaly,final_normal,final_anomaly,validacao_normal,validacao_anomaly,removidos_tomek
0,Sem balanceamento,1,52603,3263,51142,1742,51142,1742,51142,1742,13151,816,0
1,ROS,1,52603,3263,51142,1742,51142,51142,51142,51142,13151,816,0
2,RUS,1,52603,3263,51142,1742,1742,1742,1742,1742,13151,816,0
3,SMOTENC + Tomek,1,52603,3263,51142,1742,51142,51142,51116,51116,13151,816,52
8,Sem balanceamento,2,52602,3263,51128,1751,51128,1751,51128,1751,13152,816,0
9,ROS,2,52602,3263,51128,1751,51128,51128,51128,51128,13152,816,0
10,RUS,2,52602,3263,51128,1751,1751,1751,1751,1751,13152,816,0
11,SMOTENC + Tomek,2,52602,3263,51128,1751,51128,51128,51110,51110,13152,816,36
16,Sem balanceamento,3,52604,3263,51125,1739,51125,1739,51125,1739,13150,816,0
17,ROS,3,52604,3263,51125,1739,51125,51125,51125,51125,13150,816,0


,modelo,estrategia,f1_macro_media
0,XGBoost,Sem balanceamento,0.870542
1,Árvore,Sem balanceamento,0.871753


### Resultados do experimento inicial — Matheus de Almeida Moreira

**Resultado observado.** Foram executadas 40 avaliações (quatro estratégias × dois modelos × cinco dobras), com grupos disjuntos e validação intacta. F1-macro médio ± desvio padrão:

| Estratégia | Árvore de Decisão | XGBoost |
|---|---:|---:|
| Sem balanceamento | **0,8718 ± 0,0040** | **0,8705 ± 0,0042** |
| RandomOverSampler | 0,8668 ± 0,0033 | 0,8597 ± 0,0041 |
| RandomUnderSampler | 0,7991 ± 0,0078 | 0,8424 ± 0,0033 |
| SMOTENC + Tomek Links | 0,8633 ± 0,0041 | 0,8619 ± 0,0025 |

Nenhum dos três balanceamentos superou a referência sem balanceamento em nenhuma das cinco dobras, para os dois modelos. **Neste primeiro experimento, a proposta era manter ausência de balanceamento. A conclusão atualizada da busca com outras proporções e pesos está na seção 2.6.** A diferença entre árvore e XGBoost sem balanceamento é pequena diante da variação entre dobras. Este experimento não estabelece superioridade geral de um algoritmo.

**XGBoost + SMOTENC/Tomek.** O recall de `anomaly` subiu de **0,7507 para 0,7911**, mas a precisão caiu de **0,7617 para 0,6969**, indicando maior cobertura de anomalias acompanhada de mais falsos positivos. O F1 da classe caiu de 0,7561 para 0,7410, e o F1-macro de 0,8705 para 0,8619 (Δ = −0,0087). Portanto, a combinação não é a escolha pelo critério F1-macro definido pelo trabalho. Se o objetivo operacional priorizasse recall, seria necessário definir o custo dos erros e validar esse critério separadamente. O custo médio também subiu: aproximadamente 15,91 s por dobra, contra 0,83 s sem balanceamento, incluindo ajuste e predição nesta máquina.

**Distribuição e efeitos dos métodos.** Na primeira dobra, o treino original tem 52.603 registros `normal` e 3.263 `anomaly`. Após remover grupos conflitantes apenas no treino, restam **51.142 normais e 1.742 anomalias** (3,29% de anomalias). A validação continua com **13.151 normais e 816 anomalias** em todas as estratégias.

- **ROS:** passa para 51.142 registros de cada classe, acrescentando 49.400 cópias de anomalias. Preserva os registros da maioria, mas não cria informação nova e aumenta o peso de poucos exemplos minoritários. Pode favorecer sobreajuste. Os resultados não mostram ganho de F1-macro, mas não isolam sua causa.
- **RUS:** mantém 1.742 registros de cada classe. Descarta 49.400 normais, cerca de **96,6% da maioria** naquela dobra. É rápido, mas essa perda de cobertura pode prejudicar a identificação de tráfego normal. No XGBoost, o recall de anomalias cresce para 0,8137 e a precisão cai para 0,6236. A árvore apresenta a maior queda de F1-macro.
- **SMOTENC + Tomek:** gera 49.400 anomalias sintéticas na primeira dobra, respeitando os valores nominais/discretos observados no treino. Em seguida, Tomek remove 52 registros (26 de cada classe), deixando 51.116 de cada classe. Nas cinco dobras, remove entre 36 e 52 registros no total por dobra: a limpeza é pequena diante da expansão promovida pelo SMOTENC. A combinação exige mais tempo e pode interpolar vizinhos inadequados em regiões ruidosas. Remover pares de Tomek não garante eliminar todo ruído nem identificar rótulos verdadeiros.

**Integração e limites.** `montar_pipeline_m2`, `por_dobra_m2`, `distribuicoes_m2`, `tabela_m2` e `candidatos_m2` ficam disponíveis para o Paulo Gabriel de Oliveira Leite. Antes de `fit`, deve-se aplicar `filtrar_linhas_treino` somente ao treino da dobra, como no avaliador. As estatísticas de imputação e escala são aprendidas antes da reamostragem. Os amostradores são ignorados durante `predict`.

As conclusões são condicionais à remoção de conflitos, aos parâmetros fixos, à proporção 1:1 e a uma única semente. Não houve busca por uma razão de balanceamento melhor. As mesmas dobras já orientaram escolhas da parte 1, então a seleção é exploratória e pode ser otimista. Confirmar na integração com outra divisão por grupos. A referência desta seção inclui escala para distâncias e codificação dos indicadores discretos, por isso o efeito do balanceamento deve ser lido contra a referência **desta seção**, e não diretamente contra R0 da parte 1. SMOTENC + Tomek é uma adaptação para dados mistos, não a execução da classe `SMOTETomek` padrão. Confirmar com a equipe/professora se XGBoost atende ao requisito de algoritmo visto em aula antes de adotá-lo na etapa 3. O teste oficial não foi usado nesta comparação.

**Execução e uso de IA.** A parte 2 foi implementada e executada com apoio do OpenAI Codex, incluindo revisão do protocolo e das contagens. As versões utilizadas aparecem na célula 2.1 e as saídas desta seção estão salvas. Houve um aviso ambiental do `joblib` ao consultar núcleos físicos no macOS. A biblioteca usou a contagem de núcleos lógicos e a execução concluiu com todas as auditorias aprovadas. Os responsáveis devem revisar a interpretação e registrar suas contribuições efetivas antes da entrega. Não se atribui aqui autoria individual não confirmada.

### 2.4 Busca ampliada: proporções e pesos de classe

O experimento anterior testou reamostragem até 50% de anomalias. Ampliamos a busca mantendo `CONFIG_M1`, a árvore, o XGBoost, o limiar 0,5 e os demais hiperparâmetros fixos. Assim, não misturamos alterações de balanceamento com log, agrupamento de `service` ou ajustes do classificador.

São **27 configurações por modelo**:

- Referência sem balanceamento.
- ROS, RUS, SMOTENC e SMOTENC + Tomek, cada um com razões minoritária/majoritária de **0,05. 0,10. 0,25. 0,50. 1,00**. As participações minoritárias correspondentes são aproximadamente **4,8%. 9,1%. 20%. 33,3%. 50%** antes da limpeza de Tomek. `sampling_strategy=0.5` significa uma anomalia para dois normais, não 50% do total.
- Tomek isolado, removendo ambos os lados dos pares. Não força uma proporção.
- Ponderação de `anomaly` com fatores **1,5. 2. 4. √(n_normal/n_anomaly). N_normal/n_anomaly**, sempre com peso 1 para `normal`. Os fatores adaptativos usam somente os rótulos do treino da dobra **após** a remoção de conflitos. Na árvore usamos `class_weight`. No XGBoost, `scale_pos_weight`. Esses fatores alteram a importância dos erros durante o treino, sem acrescentar ou remover linhas. Não combinamos ponderação e reamostragem neste experimento.

**Seleção:** executar as 27 opções nos dois modelos nas cinco dobras originais (270 ajustes). Ordenar por F1-macro médio e selecionar, para cada modelo, os dois melhores candidatos diferentes da referência. Reavaliar esses candidatos e a referência em cinco dobras por grupos com **semente 137**, mantendo a semente dos modelos/amostradores em 42 (30 ajustes adicionais). Não ampliar a grade depois de observar os resultados.

A segunda divisão é uma **checagem de estabilidade**, não um teste independente: reutiliza os mesmos registros e também não elimina o otimismo da seleção feita anteriormente na parte 1. Uma opção só será recomendada sobre a referência se melhorar o F1-macro médio nas **duas divisões**. Entre as opções elegíveis, usamos a média das dez avaliações. As métricas são exploratórias e não substituem avaliação externa ou validação aninhada.

Um cache temporário reutiliza transformações e reamostragens idênticas apenas dentro da mesma dobra. Não compartilha ajustes entre dobras e é descartado ao fim de cada uma. Tempos com cache dependem da ordem e não serão usados para comparar desempenho computacional.

In [10]:
# Grade fixada antes da execução; todas as estratégias preservam a validação original.
from tempfile import TemporaryDirectory
from joblib import Memory

RAZOES_BUSCA_M2 = (0.05, 0.10, 0.25, 0.50, 1.00)
CANDIDATOS_BUSCA_M2 = [{"nome": "Sem balanceamento", "metodo": "nenhum"}]
for peso in (1.5, 2.0, 4.0, "raiz", "inverso"):
    CANDIDATOS_BUSCA_M2.append({"nome": f"Peso {peso}", "metodo": "peso", "peso": peso})
CANDIDATOS_BUSCA_M2.append({"nome": "Tomek isolado", "metodo": "tomek"})
for metodo in ("ROS", "RUS", "SMOTENC", "SMOTENC + Tomek"):
    for razao in RAZOES_BUSCA_M2:
        CANDIDATOS_BUSCA_M2.append({"nome": f"{metodo} r={razao:g}",
                                   "metodo": metodo, "razao": razao})
ESPECIFICACOES_BUSCA_M2 = {c["nome"]: c for c in CANDIDATOS_BUSCA_M2}
assert len(ESPECIFICACOES_BUSCA_M2) == 27


def montar_pipeline_busca_m2(candidato, nome_modelo, y_fit, memoria=None):
    """Cria pipeline novo; y_fit deve conter apenas o treino já filtrado da dobra.

    Retorna pipeline e peso efetivo. A fábrica também serve à integração posterior.
    """
    metodo = candidato["metodo"]
    contagens = contagens_m2(y_fit)
    if min(contagens.values()) <= 5:
        raise ValueError("Treino insuficiente para comparar SMOTENC com k_neighbors=5.")
    razao_original = contagens[1] / contagens[0]
    if "razao" in candidato and candidato["razao"] < razao_original:
        raise ValueError("Razão solicitada menor que a razão atual; revisar a grade.")
    modelo = clone(MODELOS_M2[nome_modelo])
    peso = 1.0
    if metodo == "peso":
        valor = candidato["peso"]
        desequilibrio = contagens[0] / contagens[1]
        peso = np.sqrt(desequilibrio) if valor == "raiz" else (
            desequilibrio if valor == "inverso" else float(valor))
        if nome_modelo == "Árvore":
            modelo.set_params(class_weight={0: 1.0, 1: peso})
        elif nome_modelo == "XGBoost":
            modelo.set_params(scale_pos_weight=peso)
        else:
            raise ValueError("Defina explicitamente a ponderação para este modelo.")
    pipe = montar_pipeline_m2("Sem balanceamento", modelo)
    amostrador = None
    if metodo == "ROS":
        amostrador = RandomOverSampler(sampling_strategy=candidato["razao"], random_state=SEMENTE)
    elif metodo == "RUS":
        amostrador = RandomUnderSampler(sampling_strategy=candidato["razao"],
                                        replacement=False, random_state=SEMENTE)
    elif metodo in ("SMOTENC", "SMOTENC + Tomek"):
        amostrador = SMOTENC(categorical_features="auto", sampling_strategy=candidato["razao"],
                            k_neighbors=5, random_state=SEMENTE)
    limpeza = TomekLinks(sampling_strategy="all", n_jobs=2) if metodo in (
        "tomek", "SMOTENC + Tomek") else None
    pipe.set_params(reamostrar__amostrador=amostrador, tomek__amostrador=limpeza,
                    memory=memoria)
    return pipe, float(peso)


def avaliar_grade_m2(candidatos_por_modelo, dobras, semente_dobras):
    """Todos os ajustes ocorrem dentro das dobras; retorna métricas e auditoria de linhas."""
    resultados, distribuicoes = [], []
    nomes = [c["nome"] for c in CANDIDATOS_BUSCA_M2
             if any(c["nome"] in v for v in candidatos_por_modelo.values())]
    with TemporaryDirectory(prefix="balanceamento_m2_") as diretorio:
        for dobra, (itr, iva) in enumerate(dobras, start=1):
            assert set(grupos_treino.iloc[itr]).isdisjoint(grupos_treino.iloc[iva])
            X_tr, y_tr = filtrar_linhas_treino(
                X_treino.iloc[itr], y_treino.iloc[itr], CONFIG_M1["linhas_treino"])
            X_va, y_va = X_treino.iloc[iva].copy(), y_treino.iloc[iva].copy()
            memoria = Memory(Path(diretorio) / f"dobra_{dobra}", verbose=0)
            for posicao, nome in enumerate(nomes):
                candidato = ESPECIFICACOES_BUSCA_M2[nome]
                for nome_modelo, selecionados in candidatos_por_modelo.items():
                    if nome not in selecionados:
                        continue
                    pipe, peso = montar_pipeline_busca_m2(candidato, nome_modelo, y_tr, memoria)
                    with threadpool_limits(limits=2):
                        pipe.fit(X_tr, y_tr)
                        pred = pipe.predict(X_va)
                    assert len(pred) == len(y_va)
                    assert X_va.equals(X_treino.iloc[iva]) and y_va.equals(y_treino.iloc[iva])
                    p, r, f, suporte = precision_recall_fscore_support(
                        y_va, pred, labels=[0, 1], zero_division=0)
                    registro = {"semente_dobras": semente_dobras, "dobra": dobra,
                                "modelo": nome_modelo, "candidato": nome, "peso_anomaly": peso,
                                "f1_macro": float(f.mean())}
                    for j, classe in enumerate(CLASSES):
                        registro.update({f"precisao_{classe}": p[j], f"recall_{classe}": r[j],
                                         f"f1_{classe}": f[j], f"suporte_{classe}": int(suporte[j])})
                    resultados.append(registro)
                    reamostrar, tomek = pipe.named_steps["reamostrar"], pipe.named_steps["tomek"]
                    assert reamostrar.antes_ == contagens_m2(y_tr)
                    assert reamostrar.depois_ == tomek.antes_
                    assert all(tomek.depois_[j] <= tomek.antes_[j] for j in (0, 1))
                    if "razao" in candidato:
                        obtida = reamostrar.depois_[1] / reamostrar.depois_[0]
                        assert abs(obtida - candidato["razao"]) <= 1 / min(reamostrar.depois_.values())
                    if candidato["metodo"] in ("nenhum", "peso"):
                        assert tomek.depois_ == contagens_m2(y_tr)
                    dist = {"semente_dobras": semente_dobras, "dobra": dobra,
                            "modelo": nome_modelo, "candidato": nome}
                    for etapa, contagens in {"original": contagens_m2(y_treino.iloc[itr]),
                                            "filtrado": contagens_m2(y_tr),
                                            "reamostrado": reamostrar.depois_,
                                            "final": tomek.depois_, "validacao": contagens_m2(y_va)}.items():
                        for j, classe in enumerate(CLASSES):
                            dist[f"{etapa}_{classe}"] = contagens[j]
                    distribuicoes.append(dist)
                # Progresso por família, para acompanhar a execução sem imprimir 270 linhas.
                proximo = ESPECIFICACOES_BUSCA_M2[nomes[posicao + 1]]["metodo"] if posicao + 1 < len(nomes) else None
                if proximo != candidato["metodo"]:
                    print(f"Semente {semente_dobras}, dobra {dobra}: {candidato['metodo']} concluído.", flush=True)
            memoria.clear(warn=False)
    dados = pd.DataFrame(resultados)
    dist = pd.DataFrame(distribuicoes)
    ref = dados[dados.candidato == "Sem balanceamento"]
    dados = dados.merge(ref[["semente_dobras", "dobra", "modelo", "f1_macro"]].rename(
        columns={"f1_macro": "f1_referencia"}), on=["semente_dobras", "dobra", "modelo"], validate="many_to_one")
    dados["delta_f1"] = dados.f1_macro - dados.f1_referencia
    assert dist.groupby(["semente_dobras", "dobra"])[["validacao_normal", "validacao_anomaly"]].nunique().eq(1).all().all()
    cols_contagens = [c for c in dist if c not in ("semente_dobras", "dobra", "modelo", "candidato")]
    assert dist.groupby(["semente_dobras", "dobra", "candidato"])[cols_contagens].nunique().eq(1).all().all()
    return dados, dist


def resumir_grade_m2(dados):
    return dados.groupby(["modelo", "candidato"], sort=False).agg(
        f1_macro_media=("f1_macro", "mean"), f1_macro_dp=("f1_macro", "std"),
        delta_f1=("delta_f1", "mean"), dobras_melhores=("delta_f1", lambda s: int((s > 0).sum())),
        precisao_anomaly=("precisao_anomaly", "mean"), recall_anomaly=("recall_anomaly", "mean"),
        f1_anomaly=("f1_anomaly", "mean"), f1_normal=("f1_normal", "mean"),
    ).reset_index().sort_values(["modelo", "f1_macro_media", "candidato"], ascending=[True, False, True])

In [11]:
# Busca principal: mesmas cinco dobras da parte 1, modelo e limiar fixos.
grade_por_modelo_m2 = {nome: list(ESPECIFICACOES_BUSCA_M2) for nome in MODELOS_M2}
busca_por_dobra_m2, busca_distribuicoes_m2 = avaliar_grade_m2(
    grade_por_modelo_m2, INDICES_DOBRAS, SEMENTE)
assert len(busca_por_dobra_m2) == 270
tabela_busca_m2 = resumir_grade_m2(busca_por_dobra_m2)
with pd.option_context("display.max_rows", 60, "display.max_columns", None, "display.float_format", "{:.4f}".format):
    display(tabela_busca_m2)

# A lista é congelada antes de executar a segunda divisão.
finalistas_m2 = {}
for nome_modelo in MODELOS_M2:
    ranking = tabela_busca_m2[(tabela_busca_m2.modelo == nome_modelo)
                              & (tabela_busca_m2.candidato != "Sem balanceamento")]
    finalistas_m2[nome_modelo] = ["Sem balanceamento"] + ranking.head(2).candidato.tolist()
print("Finalistas para a segunda divisão:", finalistas_m2)

Semente 42, dobra 1: nenhum concluído.
Semente 42, dobra 1: peso concluído.
Semente 42, dobra 1: tomek concluído.
Semente 42, dobra 1: ROS concluído.
Semente 42, dobra 1: RUS concluído.
Semente 42, dobra 1: SMOTENC concluído.
Semente 42, dobra 1: SMOTENC + Tomek concluído.
Semente 42, dobra 2: nenhum concluído.
Semente 42, dobra 2: peso concluído.
Semente 42, dobra 2: tomek concluído.
Semente 42, dobra 2: ROS concluído.
Semente 42, dobra 2: RUS concluído.
Semente 42, dobra 2: SMOTENC concluído.
Semente 42, dobra 2: SMOTENC + Tomek concluído.
Semente 42, dobra 3: nenhum concluído.
Semente 42, dobra 3: peso concluído.
Semente 42, dobra 3: tomek concluído.
Semente 42, dobra 3: ROS concluído.
Semente 42, dobra 3: RUS concluído.
Semente 42, dobra 3: SMOTENC concluído.
Semente 42, dobra 3: SMOTENC + Tomek concluído.
Semente 42, dobra 4: nenhum concluído.
Semente 42, dobra 4: peso concluído.
Semente 42, dobra 4: tomek concluído.
Semente 42, dobra 4: ROS concluído.
Semente 42, dobra 4: RUS con

,modelo,candidato,f1_macro_media,f1_macro_dp,delta_f1,dobras_melhores,precisao_anomaly,recall_anomaly,f1_anomaly,f1_normal
9,XGBoost,Peso raiz,0.8743,0.0018,0.0038,3,0.7400,0.7894,0.7638,0.9848
17,XGBoost,ROS r=0.1,0.8736,0.0030,0.0030,4,0.7505,0.7742,0.7621,0.9850
7,XGBoost,Peso 4.0,0.8734,0.0024,0.0028,3,0.7433,0.7816,0.7619,0.9848
19,XGBoost,ROS r=0.25,0.8729,0.0026,0.0024,3,0.7314,0.7938,0.7613,0.9845
5,XGBoost,Peso 2.0,0.8724,0.0028,0.0019,4,0.7545,0.7654,0.7599,0.9850
3,XGBoost,Peso 1.5,0.8723,0.0028,0.0017,4,0.7580,0.7610,0.7595,0.9850
37,XGBoost,SMOTENC r=0.1,0.8722,0.0036,0.0017,4,0.7513,0.7681,0.7596,0.9849
27,XGBoost,RUS r=0.1,0.8719,0.0034,0.0013,3,0.7438,0.7749,0.7590,0.9847
47,XGBoost,SMOTENC + Tomek r=0.1,0.8716,0.0036,0.0010,4,0.7517,0.7651,0.7583,0.9849
25,XGBoost,RUS r=0.05,0.8714,0.0030,0.0009,3,0.7575,0.7583,0.7579,0.9850


### 2.5 Estabilidade dos finalistas em outra divisão por grupos

A lista de finalistas vem exclusivamente da busca anterior. Esta etapa usa a semente 137 para criar novas dobras de `StratifiedGroupKFold`. Nenhum registro do teste oficial participa. Recalculamos filtros de conflitos, estatísticas, pesos adaptativos e reamostragem a cada ajuste.

In [12]:
SEMENTE_CONFIRMACAO_M2 = 137
dobras_confirmacao_m2 = list(StratifiedGroupKFold(
    n_splits=N_DOBRAS, shuffle=True, random_state=SEMENTE_CONFIRMACAO_M2
).split(X_treino, y_treino, grupos_treino))
confirmacao_por_dobra_m2, confirmacao_distribuicoes_m2 = avaliar_grade_m2(
    finalistas_m2, dobras_confirmacao_m2, SEMENTE_CONFIRMACAO_M2)
assert len(confirmacao_por_dobra_m2) == 30
confirmacao_m2 = resumir_grade_m2(confirmacao_por_dobra_m2)
comparacao_sementes_m2 = tabela_busca_m2.merge(
    confirmacao_m2, on=["modelo", "candidato"], suffixes=("_42", "_137"), validate="one_to_one")
ambas_divisoes_m2 = pd.concat([busca_por_dobra_m2, confirmacao_por_dobra_m2], ignore_index=True)

# Aplicar a regra definida antes da busca: melhorar a média nas duas divisões.
recomendacoes_lista_m2 = []
for nome_modelo in MODELOS_M2:
    comp = comparacao_sementes_m2[comparacao_sementes_m2.modelo == nome_modelo].copy()
    comp["f1_media_duas_divisoes"] = (comp.f1_macro_media_42 + comp.f1_macro_media_137) / 2
    elegiveis = comp[(comp.candidato == "Sem balanceamento")
                    | ((comp.delta_f1_42 > 0) & (comp.delta_f1_137 > 0))]
    melhor = elegiveis.sort_values(["f1_media_duas_divisoes", "candidato"], ascending=[False, True]).iloc[0]
    recomendacoes_lista_m2.append({"modelo": nome_modelo, "candidato": melhor.candidato,
                                  "f1_media_duas_divisoes": melhor.f1_media_duas_divisoes,
                                  "delta_42": melhor.delta_f1_42, "delta_137": melhor.delta_f1_137})
recomendacoes_busca_m2 = pd.DataFrame(recomendacoes_lista_m2)
CONFIGS_BALANCEAMENTO_M2 = {r["modelo"]: dict(ESPECIFICACOES_BUSCA_M2[r["candidato"]])
                           for r in recomendacoes_lista_m2}

with pd.option_context("display.max_columns", None, "display.float_format", "{:.4f}".format):
    display(comparacao_sementes_m2[["modelo", "candidato", "f1_macro_media_42", "delta_f1_42",
                                  "dobras_melhores_42", "f1_macro_media_137", "delta_f1_137",
                                  "dobras_melhores_137", "precisao_anomaly_137", "recall_anomaly_137"]])
    display(recomendacoes_busca_m2)

# Evidências por dobra dos candidatos escolhidos; inclui pesos efetivos e distribuição final.
escolhas_m2 = recomendacoes_busca_m2[["modelo", "candidato"]]
metricas_escolhidas_m2 = ambas_divisoes_m2.merge(escolhas_m2, on=["modelo", "candidato"], validate="many_to_one")
distribuicoes_escolhidas_m2 = pd.concat([busca_distribuicoes_m2, confirmacao_distribuicoes_m2]).merge(
    escolhas_m2, on=["modelo", "candidato"], validate="many_to_one")
with pd.option_context("display.max_rows", 25, "display.max_columns", None, "display.float_format", "{:.4f}".format):
    display(metricas_escolhidas_m2[["semente_dobras", "dobra", "modelo", "candidato", "peso_anomaly",
                                  "f1_macro", "delta_f1", "precisao_anomaly", "recall_anomaly"]])
    display(distribuicoes_escolhidas_m2[["semente_dobras", "dobra", "modelo", "candidato",
                                       "filtrado_normal", "filtrado_anomaly", "final_normal", "final_anomaly",
                                       "validacao_normal", "validacao_anomaly"]])
print("Configurações propostas para integração:", CONFIGS_BALANCEAMENTO_M2)
print("300 ajustes concluídos; auditorias de grupos, contagens e validação aprovadas.")

Semente 137, dobra 1: nenhum concluído.
Semente 137, dobra 1: peso concluído.
Semente 137, dobra 1: ROS concluído.
Semente 137, dobra 1: SMOTENC + Tomek concluído.
Semente 137, dobra 2: nenhum concluído.
Semente 137, dobra 2: peso concluído.
Semente 137, dobra 2: ROS concluído.
Semente 137, dobra 2: SMOTENC + Tomek concluído.
Semente 137, dobra 3: nenhum concluído.
Semente 137, dobra 3: peso concluído.
Semente 137, dobra 3: ROS concluído.
Semente 137, dobra 3: SMOTENC + Tomek concluído.
Semente 137, dobra 4: nenhum concluído.
Semente 137, dobra 4: peso concluído.
Semente 137, dobra 4: ROS concluído.
Semente 137, dobra 4: SMOTENC + Tomek concluído.
Semente 137, dobra 5: nenhum concluído.
Semente 137, dobra 5: peso concluído.
Semente 137, dobra 5: ROS concluído.
Semente 137, dobra 5: SMOTENC + Tomek concluído.
Configurações propostas para integração: {'Árvore': {'nome': 'Peso 2.0', 'metodo': 'peso', 'peso': 2.0}, 'XGBoost': {'nome': 'Peso raiz', 'metodo': 'peso', 'peso': 'raiz'}}
300 aju

,modelo,candidato,f1_macro_media_42,delta_f1_42,dobras_melhores_42,f1_macro_media_137,delta_f1_137,dobras_melhores_137,precisao_anomaly_137,recall_anomaly_137
0,XGBoost,Peso raiz,0.8743,0.0038,3,0.8744,0.0045,5,0.7417,0.7882
1,XGBoost,ROS r=0.1,0.8736,0.0030,4,0.8734,0.0036,5,0.7489,0.7759
2,XGBoost,Sem balanceamento,0.8705,0.0000,0,0.8698,0.0000,0,0.7622,0.7480
3,Árvore,Peso 2.0,0.8731,0.0013,4,0.8711,0.0002,3,0.7417,0.7750
4,Árvore,SMOTENC + Tomek r=0.05,0.8719,0.0002,2,0.8719,0.0010,4,0.7409,0.7789
5,Árvore,Sem balanceamento,0.8718,0.0000,0,0.8709,0.0000,0,0.7400,0.7759


,modelo,candidato,f1_media_duas_divisoes,delta_42,delta_137
0,Árvore,Peso 2.0,0.8721,0.0013,0.0002
1,XGBoost,Peso raiz,0.8744,0.0038,0.0045


,semente_dobras,dobra,modelo,candidato,peso_anomaly,f1_macro,delta_f1,precisao_anomaly,recall_anomaly
0,42,1,Árvore,Peso 2.0,2.0000,0.8721,0.0022,0.7438,0.7757
1,42,1,XGBoost,Peso raiz,5.4183,0.8733,0.0063,0.7355,0.7904
2,42,2,Árvore,Peso 2.0,2.0000,0.8734,0.0018,0.7388,0.7868
3,42,2,XGBoost,Peso raiz,5.4036,0.8772,0.0058,0.7457,0.7941
4,42,3,Árvore,Peso 2.0,2.0000,0.8776,-0.0009,0.7556,0.7843
5,42,3,XGBoost,Peso raiz,5.4221,0.8750,-0.0007,0.7361,0.7966
6,42,4,Árvore,Peso 2.0,2.0000,0.8703,0.0023,0.7530,0.7586
7,42,4,XGBoost,Peso raiz,5.4139,0.8737,0.0080,0.7506,0.7745
8,42,5,Árvore,Peso 2.0,2.0000,0.8720,0.0010,0.7415,0.7779
9,42,5,XGBoost,Peso raiz,5.3981,0.8726,-0.0005,0.7321,0.7914


,semente_dobras,dobra,modelo,candidato,filtrado_normal,filtrado_anomaly,final_normal,final_anomaly,validacao_normal,validacao_anomaly
0,42,1,Árvore,Peso 2.0,51142,1742,51142,1742,13151,816
1,42,1,XGBoost,Peso raiz,51142,1742,51142,1742,13151,816
2,42,2,Árvore,Peso 2.0,51128,1751,51128,1751,13152,816
3,42,2,XGBoost,Peso raiz,51128,1751,51128,1751,13152,816
4,42,3,Árvore,Peso 2.0,51125,1739,51125,1739,13150,816
5,42,3,XGBoost,Peso raiz,51125,1739,51125,1739,13150,816
6,42,4,Árvore,Peso 2.0,51146,1745,51146,1745,13150,816
7,42,4,XGBoost,Peso raiz,51146,1745,51146,1745,13150,816
8,42,5,Árvore,Peso 2.0,51139,1755,51139,1755,13151,815
9,42,5,XGBoost,Peso raiz,51139,1755,51139,1755,13151,815


### 2.6 Conclusão da busca ampliada

**A busca ampliada muda a recomendação do experimento inicial.** O melhor resultado entre os candidatos avaliados e verificados em duas divisões foi **XGBoost com ponderação moderada da classe `anomaly`**, usando `scale_pos_weight = sqrt(n_normal / n_anomaly)`. O peso é recalculado somente com o treino filtrado de cada dobra. Variou de **5,395 a 5,425**, com média **5,411**. Não é um peso fixo a ser copiado para qualquer base e não exige criar ou descartar linhas.

Foram realizados **300 novos ajustes**: 270 na busca de 27 configurações em dois modelos e cinco dobras. Mais 30 para as referências e os dois melhores candidatos de cada modelo na segunda divisão. O experimento anterior de 40 ajustes foi preservado como histórico. A tabela abaixo resume os finalistas:

| Modelo | Candidato | F1-macro, semente 42 | F1-macro, semente 137 | Média das duas divisões |
|---|---|---:|---:|---:|
| XGBoost | Peso raiz | 0.8743 | 0.8744 | 0.8744 |
| XGBoost | ROS r=0.1 | 0.8736 | 0.8734 | 0.8735 |
| XGBoost | Sem balanceamento | 0.8705 | 0.8698 | 0.8702 |
| Árvore | Peso 2.0 | 0.8731 | 0.8711 | 0.8721 |
| Árvore | SMOTENC + Tomek r=0.05 | 0.8719 | 0.8719 | 0.8719 |
| Árvore | Sem balanceamento | 0.8718 | 0.8709 | 0.8713 |

**XGBoost.** A ponderação pela raiz elevou o F1-macro médio de **0,8702 para 0,8744** nas duas divisões (ganho de **0,0042**, aproximadamente 0,42 ponto percentual). Os ganhos médios foram +0,0038 na semente 42 e +0,0045 na semente 137. Superou a referência em **8 das 10 avaliações**, incluindo todas as cinco da segunda divisão. O recall médio de anomalias subiu de **0,7493 para 0,7888**, com precisão de **0,7620 para 0,7408**. O F1 de anomalias passou de **0,7554 para 0,7639**. O ganho de cobertura compensou a perda de precisão pelo critério definido.

**Melhor alternativa de reamostragem entre os finalistas:** ROS com razão minoritária/majoritária **0,10**, isto é, aproximadamente **9,1% de anomalias**, atingiu F1-macro médio **0,8735** no XGBoost. Também melhorou nas duas divisões (+0,0030 e +0,0036), vencendo em 9 das 10 avaliações. É uma alternativa válida, mas ficou abaixo da ponderação pela raiz na média agregada. Não implica que toda reamostragem seja pior: sua intensidade importa.

**Árvore de Decisão.** A regra previamente definida selecionou peso **2,0** para anomalias, com F1-macro médio **0,8721**, contra **0,8713** sem balanceamento. O ganho médio foi de apenas **0,0007**. Na segunda divisão, caiu para +0,0002. Portanto, há uma indicação exploratória, mas o efeito é pequeno e não sustenta afirmar superioridade estatística. SMOTENC + Tomek com razão 0,05 também foi ligeiramente melhor que a referência nas duas divisões, com média 0,8719.

**A proposta de 33,3% de anomalias foi testada.** SMOTENC + Tomek com razão 0,50 obteve **0,8673** no XGBoost na divisão original, contra 0,8705 sem balanceamento. Não entrou entre os finalistas. O balanceamento 1:1 ficou em 0,8619. Já a ponderação pela razão completa (`n_normal/n_anomaly`, perto de 29) ficou em 0,8609. Esses resultados favorecem uma correção moderada neste pré-processamento, em vez de impor a mesma importância total às duas classes. Não reproduzimos o pipeline externo com agrupamento de `service` e log, portanto não avaliamos a combinação completa descrita naquele resumo.

**Proposta para integração.** Usar `CONFIGS_BALANCEAMENTO_M2` e `montar_pipeline_busca_m2`, mantendo a filtragem de conflitos somente no treino da dobra:

```python
X_fit, y_fit = filtrar_linhas_treino(X_tr, y_tr, CONFIG_M1["linhas_treino"])
pipe, peso_efetivo = montar_pipeline_busca_m2(
    CONFIGS_BALANCEAMENTO_M2["XGBoost"], "XGBoost", y_fit
)
pipe.fit(X_fit, y_fit)
pred = pipe.predict(X_va)
```

Esse trecho é um exemplo de uso, não um treinamento final nem uma previsão do teste oficial. Para os candidatos recomendados por ponderação, as quantidades de registros após o filtro permanecem inalteradas: na primeira dobra original, **51.142 normais e 1.742 anomalias**. A validação mantém **13.151 normais e 816 anomalias**. As tabelas `busca_distribuicoes_m2` e `confirmacao_distribuicoes_m2` conservam as contagens antes e depois de cada estratégia. `busca_por_dobra_m2` e `confirmacao_por_dobra_m2` conservam as métricas das duas classes em cada dobra.

**Limites da conclusão.** “Melhor” significa melhor entre os candidatos e hiperparâmetros testados, pelo critério exploratório estabelecido. As duas divisões reutilizam os mesmos registros. Suas dez avaliações não são observações independentes. Não realizamos validação aninhada, teste de significância, ajuste de limiar ou busca de hiperparâmetros do classificador. As decisões anteriores de limpeza também usaram esta base. O ganho do XGBoost é pequeno, embora tenha mantido sinal positivo nas duas divisões. Requer confirmação na integração e avaliação externa. A admissibilidade do XGBoost como algoritmo visto em aula continua pendente para a etapa 3.

**Reprodução e auditoria.** Os novos blocos foram executados em processo Python novo, com recarga dos pré-requisitos das partes 1 e 2. As saídas foram salvas no notebook. Foram verificadas separação por grupos, preservação da validação, proporções de reamostragem, validade das categorias sintéticas e igualdade das contagens entre modelos para cada estratégia. O teste oficial não foi lido nesta busca. O cache temporário foi descartado e não é necessário para reproduzir os resultados. A implementação, execução e interpretação contaram com apoio do OpenAI Codex. A equipe deve revisar a conclusão e registrar as contribuições efetivas.

## Carlos Eduardo de Melo Sabino — Codificação e transformação numérica

**Objetivo da seção:** Implementar pelo menos uma conversão simbólico-numérica e uma transformação numérica. Justificar One-Hot ou alternativa para service e tratamento de categorias desconhecidas. Comparar com uma referência. Por exemplo, sem escala versus StandardScaler/RobustScaler.

**Conteúdo apresentado:** Blocos de codificação/escala ajustados por dobra, dimensão resultante e comparação explícita com justificativa técnica.


### 3.1 Configuração dos experimentos

Usamos o carregamento, as cinco dobras e o modelo de referência definidos pelo Samuel Lucas Rodrigues Vieira. Mantemos o tratamento dos dados fixo e não aplicamos balanceamento, para comparar o efeito da codificação e da transformação numérica. Todos os ajustes serão feitos apenas no treino de cada dobra.

In [10]:
print("Dados de treino:", X_treino.shape)
print("Quantidade de dobras:", len(INDICES_DOBRAS))
print("Modelo de referência:", MODELO_REFERENCIA)
print("Configuração proposta pelo Samuel Lucas Rodrigues Vieira:")
display(pd.Series(CONFIG_M1, name="configuracao"))

config_base_m3 = CONFIG_M1.copy()

Dados de treino: (69833, 46)
Quantidade de dobras: 5
Modelo de referência: DecisionTreeClassifier(random_state=42)
Configuração proposta pelo Samuel Lucas Rodrigues Vieira:


normalizar_categorias             True
fora_dominio                   ausente
remover_redundantes               True
imputacao_num                  mediana
imputacao_cat                constante
linhas_treino            sem_conflitos
Name: configuracao, dtype: object

### 3.2 Blocos de codificação e transformação numérica

Comparamos One-Hot sem agrupamento com One-Hot agrupando categorias com menos de 100 registros no treino de cada dobra. Para os atributos numéricos, comparamos ausência de escala, StandardScaler e RobustScaler. As demais escolhas do Samuel Lucas Rodrigues Vieira são mantidas. Como a árvore costuma ser pouco sensível à escala, resultados semelhantes entre as transformações não significam que elas sejam dispensáveis para outros modelos.

In [11]:
from sklearn.preprocessing import StandardScaler, RobustScaler

def montar_preprocessamento_m3(config, codificacao="onehot", escala="sem_escala"):
    parametros = {k: v for k, v in config.items() if k != "linhas_treino"}
    prep = montar_preprocessamento(**parametros)
    bloco = prep.named_steps["imputacao_codificacao"]

    # Mantém a imputação numérica e acrescenta a escala escolhida.
    imp_num = bloco.transformers[0][1]
    escaladores = {
        "sem_escala": "passthrough",
        "standard": StandardScaler(),
        "robust": RobustScaler()
    }
    numerico = Pipeline([("imp", imp_num), ("escala", escaladores[escala])])

    # Categorias e frequências serão aprendidas no fit de cada dobra.
    categorico = bloco.transformers[1][1]
    if codificacao == "onehot":
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    elif codificacao == "onehot_raras":
        encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, sparse_output=False)
    else:
        raise ValueError("Codificação desconhecida.")

    categorico.set_params(ohe=encoder)
    novo_bloco = ColumnTransformer([
        ("num", numerico, _eh_numerica),
        ("cat", categorico, _eh_categorica)
    ], verbose_feature_names_out=False)

    prep.set_params(imputacao_codificacao=novo_bloco)
    return prep

print("Função de pré-processamento do Carlos Eduardo de Melo Sabino definida.")

Função de pré-processamento do Carlos Eduardo de Melo Sabino definida.


### 3.3 Comparação nas cinco dobras

Usamos One-Hot sem escala como referência. Alteramos um fator por vez: agrupamento de categorias raras ou escala numérica. Registramos F1-macro, F1 por classe, quantidade de colunas e tempo de execução. A validação permanece intacta e nenhum balanceamento é aplicado.

In [12]:
experimentos_m3 = {
    "One-Hot sem escala": ("onehot", "sem_escala"),
    "One-Hot com categorias raras agrupadas": ("onehot_raras", "sem_escala"),
    "One-Hot + StandardScaler": ("onehot", "standard"),
    "One-Hot + RobustScaler": ("onehot", "robust")
}

resultados_m3 = []

for nome, (codificacao, escala) in experimentos_m3.items():
    print(f"Executando: {nome}")

    for dobra, (itr, iva) in enumerate(INDICES_DOBRAS, start=1):
        X_tr, y_tr = filtrar_linhas_treino(
            X_treino.iloc[itr], y_treino.iloc[itr], config_base_m3["linhas_treino"]
        )
        X_va, y_va = X_treino.iloc[iva], y_treino.iloc[iva]

        prep = montar_preprocessamento_m3(config_base_m3, codificacao, escala)
        pipeline_m3 = Pipeline([("prep", prep), ("modelo", clone(MODELO_REFERENCIA))])

        inicio = time.perf_counter()
        pipeline_m3.fit(X_tr, y_tr)
        previsoes = pipeline_m3.predict(X_va)
        segundos = time.perf_counter() - inicio

        f1_normal, f1_anomaly = f1_score(y_va, previsoes, labels=[0, 1], average=None, zero_division=0)
        n_colunas = pipeline_m3.named_steps["prep"].transform(X_va.head(1)).shape[1]

        resultados_m3.append({
            "experimento": nome, "dobra": dobra,
            "f1_macro": (f1_normal + f1_anomaly) / 2,
            "f1_normal": f1_normal, "f1_anomaly": f1_anomaly,
            "n_colunas": n_colunas, "segundos": segundos
        })

por_dobra_m3 = pd.DataFrame(resultados_m3)

# Compara cada resultado com a referência na mesma dobra.
referencia_m3 = por_dobra_m3[
    por_dobra_m3["experimento"] == "One-Hot sem escala"
].set_index("dobra")["f1_macro"]

por_dobra_m3["delta_vs_referencia"] = (
    por_dobra_m3["f1_macro"] - por_dobra_m3["dobra"].map(referencia_m3)
)

tabela_m3 = por_dobra_m3.groupby("experimento", sort=False).agg(
    f1_macro_media=("f1_macro", "mean"),
    f1_macro_dp=("f1_macro", "std"),
    f1_normal=("f1_normal", "mean"),
    f1_anomaly=("f1_anomaly", "mean"),
    delta_vs_referencia=("delta_vs_referencia", "mean"),
    colunas_media=("n_colunas", "mean"),
    colunas_min=("n_colunas", "min"),
    colunas_max=("n_colunas", "max"),
    segundos_media=("segundos", "mean")
)

display(tabela_m3.round(4))
display(por_dobra_m3.pivot(index="experimento", columns="dobra", values="f1_macro").round(4))

Executando: One-Hot sem escala
Executando: One-Hot com categorias raras agrupadas
Executando: One-Hot + StandardScaler
Executando: One-Hot + RobustScaler


,f1_macro_media,f1_macro_dp,f1_normal,f1_anomaly,delta_vs_referencia,colunas_media,colunas_min,colunas_max,segundos_media
experimento,,,,,,,,,
One-Hot sem escala,0.8706,0.0044,0.9845,0.7567,0.0000,117.0,117,117,1.3021
One-Hot com categorias raras agrupadas,0.8730,0.0038,0.9848,0.7612,0.0024,65.8,65,66,0.8046
One-Hot + StandardScaler,0.8704,0.0045,0.9845,0.7562,-0.0003,117.0,117,117,1.2673
One-Hot + RobustScaler,0.8706,0.0043,0.9845,0.7566,-0.0000,117.0,117,117,1.2787


dobra,1,2,3,4,5
experimento,,,,,
One-Hot + RobustScaler,0.8685,0.8699,0.8779,0.8666,0.8701
One-Hot + StandardScaler,0.8682,0.8697,0.8779,0.8659,0.8701
One-Hot com categorias raras agrupadas,0.8715,0.8746,0.8788,0.8692,0.8709
One-Hot sem escala,0.8685,0.8703,0.8779,0.8664,0.8701


### 3.4 Categorias desconhecidas e limitações

No One-Hot comum, categorias desconhecidas são ignoradas e representadas por zeros nas colunas daquele atributo. Na alternativa com agrupamento, elas entram no grupo de categorias pouco frequentes, quando esse grupo existe. Caso contrário, também são representadas por zeros. O limite de 100 registros é aplicado separadamente a cada atributo categórico e considera apenas o treino da dobra. O agrupamento pode incluir a categoria de ausência criada pela imputação, caso sua frequência fique abaixo desse limite. A demonstração abaixo usa valores fictícios, sem consultar o teste oficial.

O limite de 100 registros reaproveita o critério exploratório da primeira entrega e não foi escolhido buscando o melhor resultado nesta comparação. No exemplo fictício, usamos o limite de 2 apenas para demonstrar o funcionamento com poucos registros.

In [13]:
exemplo_treino_m3 = pd.DataFrame({"service": ["http", "http", "smtp", "smtp", "raro"]})
exemplo_novo_m3 = pd.DataFrame({"service": ["http", "servico_desconhecido"]})

encoders_exemplo_m3 = {
    "One-Hot comum": OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    "One-Hot com agrupamento": OneHotEncoder(
        handle_unknown="infrequent_if_exist", min_frequency=2, sparse_output=False
    )
}

for nome, encoder in encoders_exemplo_m3.items():
    encoder.fit(exemplo_treino_m3)
    transformado = encoder.transform(exemplo_novo_m3)
    print(f"\n{nome}:")
    display(pd.DataFrame(
        transformado,
        columns=encoder.get_feature_names_out(),
        index=exemplo_novo_m3["service"]
    ))


One-Hot comum:


,service_http,service_raro,service_smtp
service,,,
http,1.0,0.0,0.0
servico_desconhecido,0.0,0.0,0.0



One-Hot com agrupamento:


,service_http,service_smtp,service_infrequent_sklearn
service,,,
http,1.0,0.0,0.0
servico_desconhecido,0.0,0.0,1.0


### 3.5 Resultados e justificativa — Carlos Eduardo de Melo Sabino

O One-Hot sem escala apresentou F1-macro médio de 0,8706, com 117 colunas. Ao agrupar categorias com menos de 100 registros no treino de cada dobra, o F1-macro passou para 0,8730, com desvio padrão de 0,0038. A quantidade de colunas caiu para 65 a 66, pois as frequências variam entre os treinos das dobras. Essa alternativa superou a referência nas cinco dobras, mas a diferença média de 0,0024 é pequena e não permite afirmar que haverá ganho em dados futuros. O agrupamento fica como proposta para a integração por reduzir a dimensão e manter desempenho semelhante, com uma pequena melhora observada.

StandardScaler e RobustScaler apresentaram F1-macro de 0,8704 e 0,8706, respectivamente, sem vantagem relevante sobre a referência. Isso é esperado para árvores, que costumam ser pouco sensíveis à escala. Portanto, não há evidência de benefício dessas escalas para o modelo usado nesta comparação, mas o resultado não deve ser generalizado para modelos sensíveis à escala. O One-Hot foi escolhido por representar categorias sem criar uma ordem artificial. A demonstração com dados fictícios confirmou o tratamento das categorias desconhecidas nas duas alternativas.

Os experimentos mantiveram as escolhas de tratamento do Samuel Lucas Rodrigues Vieira, as mesmas cinco dobras e o mesmo modelo de referência, sem balanceamento. Imputação, codificação e escala foram ajustadas apenas no treino de cada dobra, mantendo a validação intacta. Os resultados por dobra estão em `por_dobra_m3`, o resumo em `tabela_m3` e a função reutilizável em `montar_preprocessamento_m3`. A combinação com balanceamento e outros modelos é avaliada na parte 5.

## Arthur Soares Rolla — Redução de dimensionalidade ou seleção de instâncias

Compara a árvore de decisão com e sem PCA, usando 25 ou 50 componentes nas mesmas cinco dobras. Em cada dobra, o PCA é ajustado apenas nos dados de treino, e são comparados o F1, o tempo e o número de atributos.


In [14]:
# Arthur Soares Rolla — PCA ajustado somente dentro do treino de cada dobra.
from sklearn.decomposition import PCA

experimentos_m4 = {
    "Sem redução": None,
    "PCA — 25 componentes": 25,
    "PCA — 50 componentes": 50,
}

resultados_m4 = []
for nome, n_componentes in experimentos_m4.items():
    print(f"Executando: {nome}")

    for dobra, (itr, iva) in enumerate(INDICES_DOBRAS, start=1):
        X_tr, y_tr = filtrar_linhas_treino(
            X_treino.iloc[itr], y_treino.iloc[itr], CONFIG_M1["linhas_treino"]
        )
        X_va, y_va = X_treino.iloc[iva], y_treino.iloc[iva]

        etapas = [("prep", montar_preprocessamento_m3(
            CONFIG_M1, codificacao="onehot", escala="standard"
        ))]
        if n_componentes is not None:
            etapas.append(("pca", PCA(
                n_components=n_componentes, svd_solver="randomized", random_state=SEMENTE
            )))
        etapas.append(("modelo", clone(MODELO_REFERENCIA)))
        pipeline_m4 = Pipeline(etapas)

        inicio = time.perf_counter()
        pipeline_m4.fit(X_tr, y_tr)
        previsoes = pipeline_m4.predict(X_va)
        segundos = time.perf_counter() - inicio

        f1_normal, f1_anomaly = f1_score(
            y_va, previsoes, labels=[0, 1], average=None, zero_division=0
        )
        n_atributos_entrada = pipeline_m4.named_steps["prep"].transform(X_va.head(1)).shape[1]
        if n_componentes is None:
            n_atributos_saida = n_atributos_entrada
            variancia_pca = np.nan
        else:
            n_atributos_saida = n_componentes
            variancia_pca = pipeline_m4.named_steps["pca"].explained_variance_ratio_.sum()

        resultados_m4.append({
            "experimento": nome,
            "dobra": dobra,
            "f1_macro": (f1_normal + f1_anomaly) / 2,
            "f1_normal": f1_normal,
            "f1_anomaly": f1_anomaly,
            "atributos_entrada": n_atributos_entrada,
            "atributos_saida": n_atributos_saida,
            "reducao_pct": 100 * (1 - n_atributos_saida / n_atributos_entrada),
            "variancia_pca": variancia_pca,
            "segundos": segundos,
        })

por_dobra_m4 = pd.DataFrame(resultados_m4)
referencia_m4 = por_dobra_m4[
    por_dobra_m4["experimento"] == "Sem redução"
].set_index("dobra")["f1_macro"]
por_dobra_m4["delta_vs_referencia"] = (
    por_dobra_m4["f1_macro"] - por_dobra_m4["dobra"].map(referencia_m4)
)

tabela_m4 = por_dobra_m4.groupby("experimento", sort=False).agg(
    f1_macro_media=("f1_macro", "mean"),
    f1_macro_dp=("f1_macro", "std"),
    f1_normal=("f1_normal", "mean"),
    f1_anomaly=("f1_anomaly", "mean"),
    delta_vs_referencia=("delta_vs_referencia", "mean"),
    atributos_entrada=("atributos_entrada", "mean"),
    atributos_saida=("atributos_saida", "mean"),
    reducao_pct=("reducao_pct", "mean"),
    variancia_pca=("variancia_pca", "mean"),
    segundos_media=("segundos", "mean"),
)

with pd.option_context("display.float_format", "{:.4f}".format):
    display(tabela_m4)
    print("F1-macro por dobra:")
    display(por_dobra_m4.pivot(index="experimento", columns="dobra", values="f1_macro"))

Executando: Sem redução
Executando: PCA — 25 componentes
Executando: PCA — 50 componentes


,f1_macro_media,f1_macro_dp,f1_normal,f1_anomaly,delta_vs_referencia,atributos_entrada,atributos_saida,reducao_pct,variancia_pca,segundos_media
experimento,,,,,,,,,,
Sem redução,0.8704,0.0045,0.9845,0.7562,0.0000,117.0000,117.0000,0.0000,NaN,1.3316
PCA — 25 componentes,0.8397,0.0056,0.9813,0.6980,-0.0307,117.0000,25.0000,78.6325,0.9067,3.8244
PCA — 50 componentes,0.8537,0.0047,0.9827,0.7246,-0.0167,117.0000,50.0000,57.2650,0.9987,10.3389


F1-macro por dobra:


dobra,1,2,3,4,5
experimento,,,,,
PCA — 25 componentes,0.8345,0.8465,0.8448,0.8352,0.8373
PCA — 50 componentes,0.8534,0.8568,0.8558,0.8456,0.8566
Sem redução,0.8682,0.8697,0.8779,0.8659,0.8701


### Resultados e justificativa — Arthur Soares Rolla

A comparação foi realizada com as mesmas cinco dobras, utilizando a Árvore de Decisão como modelo de referência, sem balanceamento. Todas as etapas de pré-processamento foram ajustadas apenas com os dados de treino de cada dobra.

| Estratégia | F1-macro médio ± DP | F1 normal | F1 anomaly | Dimensões | Redução | Variância PCA | Tempo médio por dobra |
|---|---:|---:|---:|---:|---:|---:|---:|
| Sem redução | 0,8704 ± 0,0045 | 0,9845 | 0,7562 | 117 → 117 | 0,0% | — | 1,46 s |
| PCA (25 componentes) | 0,8397 ± 0,0056 | 0,9813 | 0,6980 | 117 → 25 | 78,6% | 90,67% | 3,96 s |
| PCA (50 componentes) | 0,8537 ± 0,0047 | 0,9827 | 0,7246 | 117 → 50 | 57,3% | 99,87% | 9,39 s |

Os resultados mostram que o PCA reduziu o número de dimensões, mas também diminuiu o desempenho do modelo. Mesmo preservando 99,87% da variância com 50 componentes, o F1 da classe anomaly caiu de 0,7562 para 0,7246. Além disso, o uso de PCA aumentou o tempo de execução.

**Conclusão:** para este modelo e pré-processamento, o PCA não trouxe melhorias. Por isso, a configuração sem redução de dimensões foi a melhor escolha. Essa conclusão vale para este experimento e não significa que o PCA seja inadequado para outros modelos. A decisão foi baseada apenas na validação interna, sem utilizar o conjunto de teste oficial.

## Paulo Gabriel de Oliveira Leite — Integração e comparação das estratégias

**Objetivo da seção:** Combinar os blocos das quatro seções anteriores em pipeline sem vazamento. Definir dobras internas comuns e modelo de referência com a equipe. Executar comparações explícitas, preferencialmente alterando um fator por vez. Auditar duplicatas entre dobras e consolidar os resultados.

**Conteúdo apresentado:** Tabela estratégia × modelo de referência × métricas, resultados por dobra, discussão das escolhas e configurações para a etapa 3. Não apresentar só uma combinação final.


### 5.1 Protocolo da integração e decisão sobre os sinais operacionais

Integramos a limpeza e o filtro de conflitos da parte 1, os amostradores e modelos da parte 2, a fábrica de codificação/escala da parte 3 e o PCA da parte 4. Cada ajuste recebe um pipeline novo. O filtro usa somente os rótulos do treino da dobra. A validação permanece completa, inclusive com conflitos. Usamos as cinco `INDICES_DOBRAS`, com os mesmos grupos e sementes dos experimentos anteriores.

**Decisão desta entrega:** manter `score_ids_legado`, `tipo_alerta_soc` e `tempo_resposta_analista_min` fora dos preditores, assim como `id` e `class`. A associação observada na etapa 1 e a disponibilidade temporal ainda não confirmada justificam exclusão preventiva, não uma afirmação de vazamento comprovado. A origem desses sinais continua pendente de confirmação com a professora. Esta seção não lê o teste oficial.

A grade abaixo fica definida antes da execução: R0 original, limpeza M1, agrupamento de categorias raras, escala, pesos moderados, agrupamento com pesos, ROS e RUS com razão anomaly/normal de 0,10 e PCA com 25 ou 50 componentes. A árvore é a referência e o XGBoost é complementar, com os mesmos hiperparâmetros da parte 2. Usamos peso 2 na árvore e raiz da razão normal/anomaly no XGBoost, propostas documentadas em 2.6. O peso adaptativo é recalculado no treino filtrado de cada dobra.

**Comparações controladas:** cada candidato declara seu comparador. ROS/RUS e pesos são comparados com o mesmo pré-processamento sem balanceamento. PCA usa como controle o mesmo agrupamento e StandardScaler. R0 → M1 altera um conjunto de escolhas, cujo efeito individual já foi investigado na parte 1. Não atribuímos esse ganho a um único fator. ROS/RUS apenas copiam ou selecionam linhas após a codificação. Não geram categorias fracionárias. A imputação, as frequências e a escala são aprendidas antes da reamostragem. SMOTENC já foi avaliado na parte 2 e não é aplicado sobre One-Hot nesta integração.

A representação da parte 3 mantém os indicadores discretos como numéricos, enquanto a parte 2 os tratava como categorias. Por isso, os números da parte 2 não são transpostos para esta tabela: todas as combinações são novamente treinadas com uma representação comum.

**Seleção exploratória:** ordenar por F1-macro médio na divisão 42, selecionar os dois melhores candidatos além de R0 e M1 por modelo e reavaliá-los com M1 na divisão 137. Só recomendar uma alternativa se superar M1 nas médias das duas divisões. Entre as elegíveis, usar a média das duas, com desempate pelo nome. Se nenhuma melhorar, manter M1. Não ampliar a grade após os resultados. As duas divisões reutilizam registros e já participaram de decisões anteriores. Não constituem validação independente ou aninhada. Esta etapa seleciona configurações para comparação na etapa 3, não o classificador final.

In [8]:
# Paulo Gabriel de Oliveira Leite — grade fixa e fábrica reutilizável.
from sklearn.decomposition import PCA
from copy import deepcopy

BASE_M5 = dict(limpeza="M1", codificacao="onehot", escala="sem_escala",
               balanceamento="nenhum", pca=None)
ESTRATEGIAS_M5 = {
    "R0 original": {**BASE_M5, "limpeza": "R0", "comparador": "R0 original"},
    "M1 limpeza": {**BASE_M5, "comparador": "R0 original"},
    "M3 raras": {**BASE_M5, "codificacao": "onehot_raras", "comparador": "M1 limpeza"},
    "M3 raras + escala": {**BASE_M5, "codificacao": "onehot_raras", "escala": "standard",
                           "comparador": "M3 raras"},
    "M2 pesos": {**BASE_M5, "balanceamento": "peso", "comparador": "M1 limpeza"},
    "Integrada pesos": {**BASE_M5, "codificacao": "onehot_raras", "balanceamento": "peso",
                        "comparador": "M3 raras"},
    "Integrada ROS": {**BASE_M5, "codificacao": "onehot_raras", "balanceamento": "ROS",
                      "comparador": "M3 raras"},
    "Integrada RUS": {**BASE_M5, "codificacao": "onehot_raras", "balanceamento": "RUS",
                      "comparador": "M3 raras"},
    "Integrada PCA25": {**BASE_M5, "codificacao": "onehot_raras", "escala": "standard",
                        "pca": 25, "comparador": "M3 raras + escala"},
    "Integrada PCA50": {**BASE_M5, "codificacao": "onehot_raras", "escala": "standard",
                        "pca": 50, "comparador": "M3 raras + escala"},
}
PESOS_PROPOSTOS_M5 = {"Árvore": 2.0, "XGBoost": "raiz"}
RAZAO_REAMOSTRAGEM_M5 = 0.10


def montar_pipeline_m5(config, nome_modelo, y_fit):
    """Recebe apenas y do treino filtrado. Retorna pipeline novo e peso efetivo.

    Antes do fit, o chamador deve aplicar filtrar_linhas_treino com a limpeza
    especificada. Durante predict, nenhum registro é filtrado ou reamostrado.
    """
    limpeza = dict(REFERENCIA if config["limpeza"] == "R0" else CONFIG_M1)
    modelo = clone(MODELOS_M2[nome_modelo])
    contagens = contagens_m2(y_fit)
    if min(contagens.values()) == 0:
        raise ValueError("O treino precisa conter ambas as classes.")
    peso = 1.0
    metodo = config["balanceamento"]
    if metodo == "peso":
        valor = PESOS_PROPOSTOS_M5[nome_modelo]
        peso = float(np.sqrt(contagens[0] / contagens[1]) if valor == "raiz" else valor)
        if nome_modelo == "Árvore":
            modelo.set_params(class_weight={0: 1.0, 1: peso})
        else:
            modelo.set_params(scale_pos_weight=peso)
    amostrador = None
    if metodo in {"ROS", "RUS"}:
        if contagens[1] / contagens[0] > RAZAO_REAMOSTRAGEM_M5:
            raise ValueError("Razão 0,10 incompatível com este treino; revisar configuração.")
        classe = RandomOverSampler if metodo == "ROS" else RandomUnderSampler
        amostrador = classe(sampling_strategy=RAZAO_REAMOSTRAGEM_M5, random_state=SEMENTE)
    elif metodo not in {"nenhum", "peso"}:
        raise ValueError(metodo)
    prep = montar_preprocessamento_m3(limpeza, config["codificacao"], config["escala"])
    # imblearn não aceita um Pipeline como etapa intermediária direta.
    # Incorporamos as etapas da fábrica M3, preservando a ordem e seus ajustes.
    etapas = list(prep.steps) + [("reamostrar", AmostradorAuditadoM2(amostrador))]
    if config["pca"] is not None:
        etapas.append(("pca", PCA(n_components=config["pca"], svd_solver="randomized",
                                   random_state=SEMENTE)))
    etapas.append(("modelo", modelo))
    return PipelineBalanceado(etapas), peso


assert not set(SINAIS_OPERACIONAIS + [COL_ID, COL_ROTULO]).intersection(X_treino.columns)
display(pd.DataFrame(ESTRATEGIAS_M5).T.rename_axis("estrategia"))
print("Ambiente:", {"Python": platform.python_version(), "numpy": np.__version__,
                   "pandas": pd.__version__, "sklearn": sklearn.__version__,
                   "imblearn": imblearn.__version__, "xgboost": xgboost.__version__})

Ambiente: {'Python': '3.13.14', 'numpy': '2.5.2', 'pandas': '3.0.5', 'sklearn': '1.9.0', 'imblearn': '0.14.2', 'xgboost': '3.4.1'}


,limpeza,codificacao,escala,balanceamento,pca,comparador
estrategia,,,,,,
R0 original,R0,onehot,sem_escala,nenhum,NaN,R0 original
M1 limpeza,M1,onehot,sem_escala,nenhum,NaN,R0 original
M3 raras,M1,onehot_raras,sem_escala,nenhum,NaN,M1 limpeza
M3 raras + escala,M1,onehot_raras,standard,nenhum,NaN,M3 raras
M2 pesos,M1,onehot,sem_escala,peso,NaN,M1 limpeza
Integrada pesos,M1,onehot_raras,sem_escala,peso,NaN,M3 raras
Integrada ROS,M1,onehot_raras,sem_escala,ROS,NaN,M3 raras
Integrada RUS,M1,onehot_raras,sem_escala,RUS,NaN,M3 raras
Integrada PCA25,M1,onehot_raras,standard,nenhum,25,M3 raras + escala


### 5.2 Execução, métricas e auditoria por dobra

Registramos precisão, recall e F1 de cada classe, F1-macro, dimensões e tempo. O tempo inclui filtro, ajuste e predição, sem cache, mas depende da máquina. As contagens antes/depois do filtro e da reamostragem permitem distinguir remoção de conflitos de balanceamento. As verificações exigem grupos disjuntos, cobertura de cada registro exatamente uma vez na validação de cada divisão, duas classes presentes e validação inalterada. O agrupamento cobre registros exatamente iguais após normalizar categorias. Não garante separar conexões correlacionadas nem cópias que diferem em ausências.

In [9]:
def avaliar_integracao_m5(candidatos_por_modelo, dobras, semente_dobras):
    resultados, auditorias = [], []
    cobertura = np.zeros(len(X_treino), dtype=int)
    for dobra, (itr, iva) in enumerate(dobras, start=1):
        assert len(np.unique(itr)) == len(itr) and len(np.unique(iva)) == len(iva)
        assert not np.intersect1d(itr, iva).size
        assert len(itr) + len(iva) == len(X_treino)
        grupos_comuns = len(set(grupos_treino.iloc[itr]) & set(grupos_treino.iloc[iva]))
        assert grupos_comuns == 0
        cobertura[iva] += 1
        X_va, y_va = X_treino.iloc[iva].copy(), y_treino.iloc[iva].copy()
        for nome_modelo, candidatos in candidatos_por_modelo.items():
            for nome in candidatos:
                config = ESTRATEGIAS_M5[nome]
                limpeza = REFERENCIA if config["limpeza"] == "R0" else CONFIG_M1
                inicio = time.perf_counter()
                X_fit, y_fit = filtrar_linhas_treino(
                    X_treino.iloc[itr], y_treino.iloc[itr], limpeza["linhas_treino"])
                assert X_fit.index.equals(y_fit.index)
                assert set(X_fit.index).issubset(X_treino.iloc[itr].index)
                assert set(y_fit.unique()) == set(y_va.unique()) == {0, 1}
                pipe, peso = montar_pipeline_m5(config, nome_modelo, y_fit)
                with threadpool_limits(limits=2):
                    pipe.fit(X_fit, y_fit)
                    pred = pipe.predict(X_va)
                segundos = time.perf_counter() - inicio
                assert len(pred) == len(iva) and set(np.unique(pred)).issubset({0, 1})
                assert X_va.equals(X_treino.iloc[iva]) and y_va.equals(y_treino.iloc[iva])
                # Inspeciona os transformadores já ajustados, sem reamostrar a validação.
                prep_ajustado = Pipeline([(nome, pipe.named_steps[nome]) for nome in
                    ("categorias", "dominio", "redundantes", "imputacao_codificacao")])
                z = prep_ajustado.transform(X_va)
                assert len(z) == len(iva) and np.isfinite(z).all()
                amostrador = pipe.named_steps["reamostrar"]
                assert amostrador.antes_ == contagens_m2(y_fit)
                if config["balanceamento"] in {"nenhum", "peso"}:
                    assert amostrador.depois_ == amostrador.antes_
                else:
                    razao = amostrador.depois_[1] / amostrador.depois_[0]
                    assert abs(razao - RAZAO_REAMOSTRAGEM_M5) <= 1 / amostrador.depois_[0]
                p, r, f, suporte = precision_recall_fscore_support(
                    y_va, pred, labels=[0, 1], zero_division=0)
                registro = dict(semente_dobras=semente_dobras, dobra=dobra, modelo=nome_modelo,
                                estrategia=nome, f1_macro=float(f.mean()), segundos=segundos,
                                atributos_entrada=z.shape[1], atributos_saida=config["pca"] or z.shape[1],
                                peso_anomaly=peso)
                for j, classe in enumerate(CLASSES):
                    registro.update({f"precisao_{classe}": p[j], f"recall_{classe}": r[j],
                                     f"f1_{classe}": f[j], f"suporte_{classe}": int(suporte[j])})
                resultados.append(registro)
                audit = dict(semente_dobras=semente_dobras, dobra=dobra, modelo=nome_modelo,
                             estrategia=nome, grupos_comuns=grupos_comuns,
                             validacao_intacta=True)
                for etapa, contagens in {"original": contagens_m2(y_treino.iloc[itr]),
                                        "filtrado": contagens_m2(y_fit),
                                        "reamostrado": amostrador.depois_,
                                        "validacao": contagens_m2(y_va)}.items():
                    for j, classe in enumerate(CLASSES):
                        audit[f"{etapa}_{classe}"] = contagens[j]
                auditorias.append(audit)
        print(f"Divisão {semente_dobras}: dobra {dobra}/{len(dobras)} concluída.", flush=True)
    assert (cobertura == 1).all()
    dados, audit = pd.DataFrame(resultados), pd.DataFrame(auditorias)
    assert audit.groupby("dobra")[["validacao_normal", "validacao_anomaly"]].nunique().eq(1).all().all()
    referencia = dados[dados.estrategia == "M1 limpeza"]
    dados = dados.merge(referencia[["dobra", "modelo", "f1_macro"]].rename(
        columns={"f1_macro": "f1_m1"}), on=["dobra", "modelo"], validate="many_to_one")
    dados["delta_vs_m1"] = dados.f1_macro - dados.f1_m1
    return dados, audit


def resumir_integracao_m5(dados):
    metricas = [f"{m}_{c}" for c in CLASSES for m in ("precisao", "recall", "f1")]
    return dados.groupby(["modelo", "estrategia"], sort=False).agg(
        f1_macro_media=("f1_macro", "mean"), f1_macro_dp=("f1_macro", "std"),
        delta_vs_m1=("delta_vs_m1", "mean"),
        dobras_melhores=("delta_vs_m1", lambda s: int((s > 0).sum())),
        atributos_min=("atributos_saida", "min"), atributos_max=("atributos_saida", "max"),
        segundos_media=("segundos", "mean"),
        **{m: (m, "mean") for m in metricas},
    ).reset_index()


grade_m5 = {modelo: list(ESTRATEGIAS_M5) for modelo in MODELOS_M2}
por_dobra_m5, auditoria_m5 = avaliar_integracao_m5(grade_m5, INDICES_DOBRAS, SEMENTE)
assert len(por_dobra_m5) == len(ESTRATEGIAS_M5) * len(MODELOS_M2) * N_DOBRAS
# Delta pareado contra o controle declarado: só compara o mesmo modelo e a mesma dobra.
por_dobra_m5["comparador"] = por_dobra_m5.estrategia.map(
    {nome: cfg["comparador"] for nome, cfg in ESTRATEGIAS_M5.items()})
controles_m5 = por_dobra_m5[["modelo", "dobra", "estrategia", "f1_macro"]].rename(
    columns={"estrategia": "comparador", "f1_macro": "f1_controle"})
por_dobra_m5 = por_dobra_m5.merge(controles_m5, on=["modelo", "dobra", "comparador"], validate="many_to_one")
por_dobra_m5["delta_controlado"] = por_dobra_m5.f1_macro - por_dobra_m5.f1_controle
tabela_m5 = resumir_integracao_m5(por_dobra_m5)
comparacoes_m5 = por_dobra_m5.groupby(["modelo", "estrategia", "comparador"], sort=False).agg(
    delta_controlado=("delta_controlado", "mean"),
    dobras_melhores=("delta_controlado", lambda s: int((s > 0).sum()))).reset_index()
with pd.option_context("display.max_rows", 30, "display.max_columns", None, "display.float_format", "{:.4f}".format):
    display(tabela_m5)
    display(comparacoes_m5)
    display(por_dobra_m5.pivot(index=["modelo", "estrategia"], columns="dobra", values="f1_macro"))
print("100 ajustes; cobertura, grupos, valores finitos e preservação da validação verificados.")

Divisão 42: dobra 1/5 concluída.
Divisão 42: dobra 2/5 concluída.
Divisão 42: dobra 3/5 concluída.
Divisão 42: dobra 4/5 concluída.
Divisão 42: dobra 5/5 concluída.
100 ajustes; cobertura, grupos, valores finitos e preservação da validação verificados.


,modelo,estrategia,f1_macro_media,f1_macro_dp,delta_vs_m1,dobras_melhores,atributos_min,atributos_max,segundos_media,precisao_normal,recall_normal,f1_normal,precisao_anomaly,recall_anomaly,f1_anomaly
0,Árvore,R0 original,0.8050,0.0056,-0.0656,0,159,164,1.1612,0.9726,0.9876,0.9800,0.7341,0.5519,0.6299
1,Árvore,M1 limpeza,0.8706,0.0044,0.0000,0,117,117,0.6802,0.9859,0.9832,0.9846,0.7407,0.7735,0.7566
2,Árvore,M3 raras,0.8730,0.0038,0.0024,5,65,66,0.4914,0.9863,0.9833,0.9848,0.7434,0.7801,0.7612
3,Árvore,M3 raras + escala,0.8728,0.0038,0.0022,5,65,66,0.4993,0.9863,0.9832,0.9848,0.7427,0.7801,0.7608
4,Árvore,M2 pesos,0.8729,0.0042,0.0023,5,117,117,0.5864,0.9862,0.9835,0.9848,0.7454,0.7774,0.7610
5,Árvore,Integrada pesos,0.8725,0.0040,0.0019,4,65,66,0.4755,0.9861,0.9835,0.9848,0.7445,0.7769,0.7603
6,Árvore,Integrada ROS,0.8700,0.0046,-0.0006,2,65,66,0.4797,0.9859,0.9831,0.9845,0.7393,0.7727,0.7555
7,Árvore,Integrada RUS,0.8644,0.0026,-0.0062,0,65,66,0.2933,0.9865,0.9802,0.9833,0.7107,0.7840,0.7455
8,Árvore,Integrada PCA25,0.8379,0.0065,-0.0327,0,25,25,1.5780,0.9811,0.9811,0.9811,0.6949,0.6950,0.6947
9,Árvore,Integrada PCA50,0.8552,0.0031,-0.0154,0,50,50,4.8753,0.9836,0.9822,0.9829,0.7192,0.7360,0.7275


,modelo,estrategia,comparador,delta_controlado,dobras_melhores
0,Árvore,R0 original,R0 original,0.0000,0
1,Árvore,M1 limpeza,R0 original,0.0656,5
2,Árvore,M3 raras,M1 limpeza,0.0024,5
3,Árvore,M3 raras + escala,M3 raras,-0.0002,0
4,Árvore,M2 pesos,M1 limpeza,0.0023,5
5,Árvore,Integrada pesos,M3 raras,-0.0005,1
6,Árvore,Integrada ROS,M3 raras,-0.0030,1
7,Árvore,Integrada RUS,M3 raras,-0.0086,0
8,Árvore,Integrada PCA25,M3 raras + escala,-0.0349,0
9,Árvore,Integrada PCA50,M3 raras + escala,-0.0176,0


dobra                          1      2      3      4      5
modelo  estrategia                                          
XGBoost Integrada PCA25   0.8469 0.8552 0.8570 0.8361 0.8510
        Integrada PCA50   0.8592 0.8661 0.8659 0.8517 0.8642
        Integrada ROS     0.8736 0.8767 0.8749 0.8685 0.8746
        Integrada RUS     0.8675 0.8762 0.8730 0.8698 0.8729
        Integrada pesos   0.8741 0.8791 0.8734 0.8736 0.8710
        M1 limpeza        0.8669 0.8710 0.8738 0.8663 0.8701
        M2 pesos          0.8746 0.8767 0.8740 0.8727 0.8726
        M3 raras          0.8690 0.8720 0.8751 0.8660 0.8707
        M3 raras + escala 0.8690 0.8720 0.8751 0.8660 0.8707
        R0 original       0.8595 0.8630 0.8687 0.8502 0.8627
Árvore  Integrada PCA25   0.8355 0.8479 0.8376 0.8298 0.8386
        Integrada PCA50   0.8533 0.8588 0.8544 0.8515 0.8580
        Integrada ROS     0.8650 0.8706 0.8759 0.8658 0.8727
        Integrada RUS     0.8628 0.8677 0.8650 0.8610 0.8656
        Integrada pesos   0.8697 0.8745 0.8772 0.8673 0.8739
        M1 limpeza        0.8685 0.8703 0.8779 0.8664 0.8699
        M2 pesos          0.8711 0.8751 0.8784 0.8671 0.8729
        M3 raras          0.8715 0.8746 0.8788 0.8692 0.8709
        M3 raras + escala 0.8715 0.8741 0.8788 0.8687 0.8709
        R0 original       0.7987 0.8108 0.8109 0.8012 0.8033

### 5.3 Estabilidade das propostas e configurações para a etapa 3

Os finalistas são derivados da tabela anterior, sem consulta ao teste. A tabela de estabilidade mostra separadamente as divisões 42 e 137. O desvio padrão por divisão é descritivo, não um intervalo de confiança. Não tratamos as dez avaliações como observações independentes. A função `ajustar_configuracao_m5` reúne a filtragem do treino, a construção dos transformadores e o ajuste do modelo. Dessa forma, a etapa 3 pode reproduzir a configuração incluindo o tratamento dos conflitos.

In [10]:
finalistas_m5 = {}
for modelo in MODELOS_M2:
    candidatos = tabela_m5[(tabela_m5.modelo == modelo)
                           & ~tabela_m5.estrategia.isin(["R0 original", "M1 limpeza"])]
    melhores = candidatos.sort_values(["f1_macro_media", "estrategia"], ascending=[False, True]).head(2)
    finalistas_m5[modelo] = ["M1 limpeza"] + melhores.estrategia.tolist()
print("Finalistas definidos pela divisão 42:", finalistas_m5)

SEMENTE_ESTABILIDADE_M5 = 137
dobras_estabilidade_m5 = list(StratifiedGroupKFold(
    n_splits=N_DOBRAS, shuffle=True, random_state=SEMENTE_ESTABILIDADE_M5
).split(X_treino, y_treino, grupos_treino))
confirmacao_por_dobra_m5, auditoria_confirmacao_m5 = avaliar_integracao_m5(
    finalistas_m5, dobras_estabilidade_m5, SEMENTE_ESTABILIDADE_M5)
assert len(confirmacao_por_dobra_m5) == 30
tabela_confirmacao_m5 = resumir_integracao_m5(confirmacao_por_dobra_m5)
estabilidade_m5 = tabela_m5.merge(tabela_confirmacao_m5, on=["modelo", "estrategia"],
                                  suffixes=("_42", "_137"), validate="one_to_one")
estabilidade_m5["f1_media_duas_divisoes"] = (estabilidade_m5.f1_macro_media_42
                                             + estabilidade_m5.f1_macro_media_137) / 2
escolhas_m5 = []
for modelo in MODELOS_M2:
    parte = estabilidade_m5[estabilidade_m5.modelo == modelo]
    elegiveis = parte[(parte.estrategia == "M1 limpeza")
                     | ((parte.delta_vs_m1_42 > 0) & (parte.delta_vs_m1_137 > 0))]
    escolhas_m5.append(elegiveis.sort_values(
        ["f1_media_duas_divisoes", "estrategia"], ascending=[False, True]).iloc[0])
recomendacoes_m5 = pd.DataFrame(escolhas_m5).reset_index(drop=True)
CONFIGS_ETAPA3_M5 = {
    r.modelo: {"estrategia": r.estrategia, "config": deepcopy(ESTRATEGIAS_M5[r.estrategia]),
               "sinais_excluidos": list(SINAIS_OPERACIONAIS),
               "semente_modelo": SEMENTE, "peso_proposto": PESOS_PROPOSTOS_M5[r.modelo],
               "razao_reamostragem": RAZAO_REAMOSTRAGEM_M5}
    for r in recomendacoes_m5.itertuples()
}


def ajustar_configuracao_m5(X_tr, y_tr, nome_modelo, config=None):
    """Ajusta exclusivamente o treino recebido; predict deve receber a validação intacta.

    Usar dentro das dobras na etapa 3. Não é um modelo já treinado nem autoriza
    tratar a avaliação de seleção como estimativa independente.
    """
    assert not set(SINAIS_OPERACIONAIS + [COL_ID, COL_ROTULO]).intersection(X_tr.columns)
    assert X_tr.index.equals(y_tr.index)
    cfg = deepcopy(CONFIGS_ETAPA3_M5[nome_modelo]["config"] if config is None else config)
    limpeza = REFERENCIA if cfg["limpeza"] == "R0" else CONFIG_M1
    X_fit, y_fit = filtrar_linhas_treino(X_tr, y_tr, limpeza["linhas_treino"])
    pipe, peso = montar_pipeline_m5(cfg, nome_modelo, y_fit)
    with threadpool_limits(limits=2):
        pipe.fit(X_fit, y_fit)
    return pipe, {"linhas_antes": len(X_tr), "linhas_fit": len(X_fit), "peso_anomaly": peso}


cols_estabilidade_m5 = ["modelo", "estrategia", "f1_macro_media_42", "f1_macro_dp_42", "delta_vs_m1_42",
                        "f1_macro_media_137", "f1_macro_dp_137", "delta_vs_m1_137", "f1_media_duas_divisoes"]
with pd.option_context("display.max_columns", None, "display.float_format", "{:.4f}".format):
    display(estabilidade_m5[cols_estabilidade_m5])
    display(recomendacoes_m5[cols_estabilidade_m5])
    display(confirmacao_por_dobra_m5.pivot(index=["modelo", "estrategia"], columns="dobra", values="f1_macro"))
    display(auditoria_m5[auditoria_m5.dobra == 1].drop(columns="validacao_intacta"))
display(pd.DataFrame(CONFIGS_ETAPA3_M5).T)
print("130 ajustes de comparação concluídos; configurações exploratórias disponíveis para a etapa 3.")

Finalistas definidos pela divisão 42: {'Árvore': ['M1 limpeza', 'M3 raras', 'M2 pesos'], 'XGBoost': ['M1 limpeza', 'Integrada pesos', 'M2 pesos']}
Divisão 137: dobra 1/5 concluída.
Divisão 137: dobra 2/5 concluída.
Divisão 137: dobra 3/5 concluída.
Divisão 137: dobra 4/5 concluída.
Divisão 137: dobra 5/5 concluída.
130 ajustes de comparação concluídos; configurações exploratórias disponíveis para a etapa 3.


,modelo,estrategia,f1_macro_media_42,f1_macro_dp_42,delta_vs_m1_42,f1_macro_media_137,f1_macro_dp_137,delta_vs_m1_137,f1_media_duas_divisoes
0,Árvore,M1 limpeza,0.8706,0.0044,0.0000,0.8715,0.0049,0.0000,0.8711
1,Árvore,M3 raras,0.8730,0.0038,0.0024,0.8706,0.0034,-0.0009,0.8718
2,Árvore,M2 pesos,0.8729,0.0042,0.0023,0.8721,0.0035,0.0005,0.8725
3,XGBoost,M1 limpeza,0.8696,0.0031,0.0000,0.8695,0.0044,0.0000,0.8696
4,XGBoost,M2 pesos,0.8741,0.0017,0.0045,0.8746,0.0044,0.0050,0.8744
5,XGBoost,Integrada pesos,0.8742,0.0029,0.0046,0.8743,0.0044,0.0048,0.8743


,modelo,estrategia,f1_macro_media_42,f1_macro_dp_42,delta_vs_m1_42,f1_macro_media_137,f1_macro_dp_137,delta_vs_m1_137,f1_media_duas_divisoes
0,Árvore,M2 pesos,0.8729,0.0042,0.0023,0.8721,0.0035,0.0005,0.8725
1,XGBoost,M2 pesos,0.8741,0.0017,0.0045,0.8746,0.0044,0.0050,0.8744


dobra                        1      2      3      4      5
modelo  estrategia                                        
XGBoost Integrada pesos 0.8795 0.8764 0.8718 0.8681 0.8758
        M1 limpeza      0.8690 0.8746 0.8690 0.8628 0.8722
        M2 pesos        0.8798 0.8757 0.8730 0.8681 0.8763
Árvore  M1 limpeza      0.8768 0.8758 0.8647 0.8696 0.8708
        M2 pesos        0.8742 0.8771 0.8687 0.8699 0.8703
        M3 raras        0.8744 0.8720 0.8671 0.8670 0.8727

,semente_dobras,dobra,modelo,estrategia,grupos_comuns,original_normal,original_anomaly,filtrado_normal,filtrado_anomaly,reamostrado_normal,reamostrado_anomaly,validacao_normal,validacao_anomaly
0,42,1,Árvore,R0 original,0,52603,3263,52603,3263,52603,3263,13151,816
1,42,1,Árvore,M1 limpeza,0,52603,3263,51142,1742,51142,1742,13151,816
2,42,1,Árvore,M3 raras,0,52603,3263,51142,1742,51142,1742,13151,816
3,42,1,Árvore,M3 raras + escala,0,52603,3263,51142,1742,51142,1742,13151,816
4,42,1,Árvore,M2 pesos,0,52603,3263,51142,1742,51142,1742,13151,816
5,42,1,Árvore,Integrada pesos,0,52603,3263,51142,1742,51142,1742,13151,816
6,42,1,Árvore,Integrada ROS,0,52603,3263,51142,1742,51142,5114,13151,816
7,42,1,Árvore,Integrada RUS,0,52603,3263,51142,1742,17420,1742,13151,816
8,42,1,Árvore,Integrada PCA25,0,52603,3263,51142,1742,51142,1742,13151,816
9,42,1,Árvore,Integrada PCA50,0,52603,3263,51142,1742,51142,1742,13151,816


,estrategia,config,sinais_excluidos,semente_modelo,peso_proposto,razao_reamostragem
Árvore,M2 pesos,"{'limpeza': 'M1', 'codificacao': 'onehot', 'es...","[score_ids_legado, tipo_alerta_soc, tempo_resp...",42,2.0,0.1
XGBoost,M2 pesos,"{'limpeza': 'M1', 'codificacao': 'onehot', 'es...","[score_ids_legado, tipo_alerta_soc, tempo_resp...",42,raiz,0.1


### 5.4 Verificação de reconstrução

Para cada recomendação, reconstruímos o pipeline em um novo ajuste usando apenas o treino da primeira dobra e verificamos a reprodução das métricas e contagens. São dois ajustes adicionais de verificação, sem nova seleção. O teste oficial não participa. As verificações também garantem que os objetos globais de treino e alvo não sejam alterados pela fábrica.

In [11]:
assinatura_X_m5 = pd.util.hash_pandas_object(X_treino, index=True).copy()
assinatura_y_m5 = pd.util.hash_pandas_object(y_treino, index=True).copy()
itr_m5, iva_m5 = INDICES_DOBRAS[0]
reproducao_m5 = []
for modelo, escolha in CONFIGS_ETAPA3_M5.items():
    reconstruido, info = ajustar_configuracao_m5(X_treino.iloc[itr_m5], y_treino.iloc[itr_m5], modelo)
    with threadpool_limits(limits=2):
        pred = reconstruido.predict(X_treino.iloc[iva_m5])
    p, r, f, suporte = precision_recall_fscore_support(
        y_treino.iloc[iva_m5], pred, labels=[0, 1], zero_division=0)
    esperado = por_dobra_m5[(por_dobra_m5.modelo == modelo)
                           & (por_dobra_m5.estrategia == escolha["estrategia"])
                           & (por_dobra_m5.dobra == 1)].iloc[0]
    for j, classe in enumerate(CLASSES):
        np.testing.assert_allclose([p[j], r[j], f[j]],
                                   [esperado[f"precisao_{classe}"], esperado[f"recall_{classe}"],
                                    esperado[f"f1_{classe}"]], rtol=0, atol=1e-12)
        assert suporte[j] == esperado[f"suporte_{classe}"]
    audit = auditoria_m5[(auditoria_m5.modelo == modelo)
                         & (auditoria_m5.estrategia == escolha["estrategia"])
                         & (auditoria_m5.dobra == 1)].iloc[0]
    assert info["linhas_fit"] == audit.filtrado_normal + audit.filtrado_anomaly
    assert reconstruido.named_steps["reamostrar"].depois_ == {
        0: audit.reamostrado_normal, 1: audit.reamostrado_anomaly}
    reproducao_m5.append({"modelo": modelo, "estrategia": escolha["estrategia"],
                         "f1_macro_reproduzido": float(f.mean()), **info})
assert assinatura_X_m5.equals(pd.util.hash_pandas_object(X_treino, index=True))
assert assinatura_y_m5.equals(pd.util.hash_pandas_object(y_treino, index=True))
display(pd.DataFrame(reproducao_m5))
print("Reconstrução aprovada: métricas por classe, contagens e treino original preservados.")

Reconstrução aprovada: métricas por classe, contagens e treino original preservados.


,modelo,estrategia,f1_macro_reproduzido,linhas_antes,linhas_fit,peso_anomaly
0,Árvore,M2 pesos,0.871105,55866,52884,2.000000
1,XGBoost,M2 pesos,0.874559,55866,52884,5.418322


### 5.5 Resultados e justificativa — Paulo Gabriel de Oliveira Leite

**Resultado da integração:** a regra definida em 5.1 selecionou **M1 + pesos moderados, One-Hot sem agrupamento, sem escala e sem PCA**, para os dois modelos. A árvore usa peso 2 para `anomaly`. O XGBoost usa a raiz da razão normal/anomaly, recalculada após filtrar o treino de cada dobra. O maior F1-macro médio observado entre os finalistas foi **0,8744 no XGBoost**. Isso é uma proposta exploratória, não a seleção definitiva da etapa 3.

| Modelo | Estratégia | F1-macro, divisão 42 | F1-macro, divisão 137 | Média das divisões |
|---|---|---:|---:|---:|
| Árvore | M1 limpeza | 0.8706 ± 0.0044 | 0.8715 ± 0.0049 | 0.8711 |
| Árvore | M3 raras | 0.8730 ± 0.0038 | 0.8706 ± 0.0034 | 0.8718 |
| Árvore | M2 pesos | 0.8729 ± 0.0042 | 0.8721 ± 0.0035 | 0.8725 |
| XGBoost | M1 limpeza | 0.8696 ± 0.0031 | 0.8695 ± 0.0044 | 0.8696 |
| XGBoost | M2 pesos | 0.8741 ± 0.0017 | 0.8746 ± 0.0044 | 0.8744 |
| XGBoost | Integrada pesos | 0.8742 ± 0.0029 | 0.8743 ± 0.0044 | 0.8743 |

**Limpeza e referência.** Na divisão 42, R0 → M1 aumentou o F1-macro de 0,8050 para 0,8706 na árvore e de 0,8609 para 0,8696 no XGBoost, em todas as dobras de ambos. Essa comparação mede o conjunto de decisões de limpeza. A atribuição do maior efeito à remoção de conflitos vem das comparações individuais da parte 1. O filtro não removeu nenhuma linha de validação. R0 foi novamente executada com a fábrica comum desta integração. Os valores desta tabela não foram copiados das seções anteriores.

**Categorias e escala.** Agrupar categorias raras reduziu 117 atributos para 65–66 na divisão 42. Na árvore, melhorou as cinco dobras nessa divisão (+0,0024 em média), mas ficou abaixo de M1 na divisão 137 (−0,0009). Não satisfez a regra de estabilidade. O StandardScaler não trouxe ganho: diferença média de −0,0002 na árvore e 0,0000 no XGBoost em relação ao mesmo agrupamento sem escala. Isso vale para os modelos avaliados. A etapa 3 deve rever escala para MLP ou outros modelos sensíveis a distâncias.

**Balanceamento e interação entre escolhas.** Os pesos moderados sem agrupamento melhoraram M1 nas duas divisões: ganhos médios de +0,0023 e +0,0005 na árvore (8/10 dobras), e +0,0045 e +0,0050 no XGBoost (10/10 dobras). No XGBoost, a precisão média de `anomaly` nas duas divisões passou de 0,7611 para 0,7410, enquanto o recall passou de 0,7479 para 0,7886: recuperou mais anomalias ao custo de mais alarmes falsos. No experimento com agrupamento, ROS e RUS com razão 0,10 melhoraram o XGBoost em relação ao mesmo agrupamento sem balanceamento (+0,0031 e +0,0013), mas pioraram a árvore (−0,0030 e −0,0086). A integração não pode presumir que todas as escolhas isoladas terão ganhos aditivos.

**Alternativa compacta no XGBoost.** Agrupamento + pesos atingiu 0,8743 na média das divisões, contra 0,8744 sem agrupamento. A diferença não arredondada é de apenas **0,000071**, aproximadamente **0,007 ponto percentual**, com 65–66 atributos em vez de 116–118 nas duas divisões. O desempate não foi alterado depois dos resultados: a regra numérica escolheu a maior média, mas essa diferença não demonstra superioridade estatística. A alternativa compacta merece permanecer documentada para uma decisão posterior sobre custo e dimensão.

**PCA.** Com o mesmo agrupamento e StandardScaler no controle, o PCA piorou todas as dobras dos dois modelos. Na árvore, F1-macro caiu de 0,8728 para 0,8379 (25 componentes) ou 0,8552 (50). No XGBoost, de 0,8706 para 0,8492 ou 0,8614. Nenhuma opção passou aos finalistas. Assim, a proposta desta etapa dispensa PCA. Essa decisão não generaliza para todo classificador.

**Configuração encaminhada.** Manter normalização das categorias, valores fora do domínio como ausentes, remoção de colunas derivadas/constantes, mediana numérica, categoria `__ausente__`, filtro de conflitos só no treino e One-Hot com categorias desconhecidas ignoradas. Manter cópias não conflitantes. Aplicar os pesos descritos acima, sem reamostragem nas propostas selecionadas. As constantes, medianas e categorias são reaprendidas em cada ajuste. `CONFIGS_ETAPA3_M5` registra as escolhas e `ajustar_configuracao_m5` executa o filtro e o treinamento sem alterar a entrada. O pipeline retornado recebe a validação completa em `predict`.

**Execução e verificação.** Foram concluídos **100 ajustes de comparação + 30 de estabilidade + 2 de reconstrução**, em processo Python novo, recarregando as definições necessárias das partes 1–3. As saídas das quatro células de código da parte 5 foram salvas. O código e os resultados das partes 1–4 foram preservados durante a integração. Os textos passaram por revisão editorial posterior, sem alteração dos experimentos. Essa verificação concentrou-se na integração e em suas dependências. A busca de 300 ajustes da parte 2 não foi reexecutada, portanto não se trata de uma nova execução integral do notebook. A sintaxe de todas as células e a estrutura do notebook foram verificadas. A reconstrução reproduziu as métricas por classe e as contagens da primeira dobra dos dois candidatos escolhidos. Os controles de grupos disjuntos, cobertura, validação intacta e ausência de valores não finitos passaram. Essas verificações confirmam a reprodução dos resultados, mas as métricas continuam sujeitas às limitações da seleção realizada nos mesmos dados.


### 5.6 Auditoria metodológica e passagem para a etapa 3

- **Dados e sinais:** alvo binário oficial, sem `id` ou sinais operacionais como preditores. Exclusão preventiva dos sinais mantida. Confirmar origem e momento de disponibilidade com a professora.
- **Duplicatas e conflitos:** dobras por grupos de registros equivalentes. Filtro de conflitos exclusivamente no treino. Validação com todas as linhas. Cópias aproximadas e dependências temporais/de host não são cobertas por esse agrupamento.
- **Transformações:** imputação, frequências de categorias, constantes, escala, reamostragem e PCA ajustados dentro do treino de cada dobra. Pesos adaptativos usam apenas seus rótulos filtrados. A razão de reamostragem é minoritária/majoritária, não fração do total.
- **Comparações:** tabelas por modelo, controles explícitos e deltas pareados. Precisão, recall e F1 das duas classes, dimensões e tempos disponíveis. Tempos não são benchmark de hardware. A seleção usa F1-macro, sem ajuste do limiar padrão.
- **Limitações:** decisões acumuladas nas mesmas bases e duas divisões já exploradas. Resultados sujeitos a otimismo de seleção, sem significância estatística ou independência entre dobras/divisões. Não afirmar que a integração eliminará erros ou vencerá em dados novos.
- **Etapa 3:** reutilizar `CONFIGS_ETAPA3_M5`, `montar_pipeline_m5` e `ajustar_configuracao_m5` junto das definições anteriores. Comparar ao menos três classificadores vistos em aula, reavaliando a adequação de escala/balanceamento por modelo. A integração não substitui as tarefas dos responsáveis pelos notebooks 03 e 04. Confirmar a admissibilidade do XGBoost.
- **Entrega:** nenhuma previsão ou métrica oficial é produzida aqui. Reconstrução final, CSV, relatório e apresentação pertencem às próximas etapas. A parte 1 já verifica compatibilidade de esquema com o teste. Esta seção trabalha exclusivamente com o treino.

**Uso de IA e revisão humana:** implementação, execução, verificações e redação desta integração tiveram apoio do OpenAI Codex. A atribuição da seção segue a divisão informada pela equipe. Não comprova autoria individual nem revisão humana. Paulo e os demais integrantes devem revisar as conclusões, compreender o código e registrar suas contribuições efetivas antes da entrega.

## Revisão conjunta

Cada membro deve explicar sua contribuição e revisar a integração com os demais. Conferir os requisitos da etapa, resultados reais, ausência de vazamento, reprodução e declaração de uso de IA antes de entregar.
